# ISIC-24 Modality Attendance — Post-hoc XAI & Metrics



In [1]:
import os
import tensorflow as tf
import kagglesdk.kaggle_env as kaggle_env
import keras
import keras_cv

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if not hasattr(kaggle_env, "get_web_endpoint"):
    kaggle_env.get_web_endpoint = kaggle_env.get_endpoint

print("GPUs:", gpus)

2026-08-02 02:31:42.102046: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-02 02:31:42.112941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785630702.125513  934072 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785630702.129525  934072 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-02 02:31:42.142886: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
class CFG:
    verbose = 2  # Verbosity
    seed = 49  # Random seed
    neg_sample = 0.01 # Downsample negative class
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [256, 256]  # Input image size
    epochs = 20 # Training epochs
    batch_size = 64  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [3]:
BASE_PATH = "/mnt/disk1/home/cgiovannoni/isic24-dum"
DATA_PATH = f"{BASE_PATH}/data/isic24"
PROC_PATH = f"{BASE_PATH}/models/enhanced/enhanced/preprocessed_dataset"
ART_PATH = f"{BASE_PATH}/artifacts/enhanced"
FEATURE_SPACE_PATH = f"{ART_PATH}/feature_space"
RUNS_PATH = f"{ART_PATH}/runs"
SUMMARIES_PATH = f"{ART_PATH}/summaries"

In [4]:
# Make the local package + the guard module importable (works with or without `pip install -e .`).
import sys, os
for _p in (os.path.join(BASE_PATH, "src"),
           os.path.join(BASE_PATH, "notebooks", "modality_attendance")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Shared modality-attendance custom layers. Importing registers the keras serializables,
# so keras.models.load_model deserializes every strategy checkpoint with NO custom_objects.
from isic24.mod_attendance import (
    ConstantScale, ScalarMultiply, LearnableAlpha, FusionWithGamma, GateEntropyLoss,
)
# FeatureSpace anti-drift guard (post-hoc correctness).
from feature_space_guard import assert_feature_space_canonical, assert_fs_matches_model


[FS GUARD] functions defined. `feature_space` not in scope yet — call assert_feature_space_canonical(feature_space) after cell 16, and assert_fs_matches_model(model, feature_space) after each load_model.


In [5]:
# Common libraries used across the post-hoc pipeline.
import glob, gc, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

## ▶ PARAMETERS

In [6]:
# Canonical study (timestamp dir) per strategy family. Point these elsewhere to analyse
# a different run; keep only the families you care about.
CANONICAL = {
    "vanilla":             os.path.join(RUNS_PATH, "vanilla", "20260604-210304"),
    "fixed_scaling":       os.path.join(RUNS_PATH, "fixed_scaling", "20260622-112038"),
    "learnable_scaling":   os.path.join(RUNS_PATH, "learnable_scaling", "20260619-184157"),
    "loss_regularization": os.path.join(RUNS_PATH, "loss_regularization", "20260610-095420"),
    "controlled_fusion":   os.path.join(RUNS_PATH, "controlled_fusion", "20260616-102253"),
    "adaptive_gating":     os.path.join(RUNS_PATH, "adaptive_gating", "20260629-185216"),
}
cf_study_dir = CANONICAL["controlled_fusion"]   # gamma sweep writes here
ag_study_dir = CANONICAL["adaptive_gating"]     # gate diagnostics write here

# Faithfulness budgets — TWO distinct knobs:
#   FAITH_BUDGETS = q-points the model is RE-SCORED at (the expensive/GPU knob). Keep 0.20
#     as an exact grid point so the low-budget cutoff is exact.
FAITH_BUDGETS = (0.05, 0.10, 0.20, 0.40, 0.60)
#   Q_MAX_REPORT = integration cutoff(s) defining the regime (low vs full). FREE to change:
#     re-integrates the SAVED curves, no model calls, no GPU.
Q_MAX_REPORT = [0.20, 0.60]

# --- evaluation set for the post-hoc metrics (identical for EVERY model) ---
#   "canonical" = the fold-0 set the runs were evaluated on (reproduces their numbers).
#   "custom"    = folds in EVAL_FOLDS, applied equally to all models (metric-stability test).
#   NB folds 1-4 are TRAINING data for every run, so a custom set mixes IN-SAMPLE data:
#      valid as a metric-stability stress test, NOT as held-out evaluation.
#      On a custom set only IMAGE faithfulness + the on-the-fly diagnostics run; SHAP-based
#      tabular/joint faithfulness is disabled (its saved arrays are keyed to fold-0 ordering).
EVAL_SET   = "canonical"     # "canonical" | "custom"
EVAL_FOLDS = [0, 1]          # used only when EVAL_SET == "custom"
# ---------------------------------------------------------------------------------------
# QUANTI CAMPIONI PER LA XAI — punto 4 della lista. Costi MISURATI (31/07):
#
#   array SHAP        1.2 s/campione   (148 s/run a n=128; mediana su 20 run, backfill 29/07)
#   faithfulness      0.9 s/campione   (118 s/run a n=128; mediana su 64 run, 06/07)
#   LORE              28  s/campione   (15 min/run a n=32) -> 13x piu' caro di SHAP+faith
#
# DUE VINCOLI, in ordine di importanza:
#
# 1) ⚠️ RITIRATO — avevo scritto qui che i primi N sono arricchiti 4.5x nei maligni. Falso:
#    la prevalenza vera e' 0.334 (`neg_sample=0.01` MA `pos_sample=5.0`), il prefisso a 0.406
#    su n=32 da' P=0.245, e `df.sample(frac=1.0)` rimescola prima dello split in fold.
#    Il problema VERO e' un altro: `pos_sample` usa replace=True, quindi 364 positivi unici
#    diventano 1820 righe. Su 256 campioni ci sono 222 lesioni uniche, e fra gli 85 positivi
#    solo 51. La XAI spiega la stessa lesione piu' volte -> le CI per run sono OTTIMISTICHE,
#    e l'n calcolato al punto (3) sotto, che assume indipendenza, e' un limite inferiore.
#
# 2) TETTO DURO A 128 sul lato tabellare/joint. Gli npz salvati contengono 128 righe
#    (`shap_values` (128,127), `explained_indices` 0..127) e `_load_shap_values` reindicizza
#    su di essi: mettere 256 qui NON allarga tabular/joint, li riporta a 128 in silenzio
#    (`n_faith_samples` nel json dira' 128, senza warning). Solo `methods=("image",)` onora
#    davvero N fino a 1091.
#
# QUINDI: 128 resta il default QUI perche' e' il tetto degli artefatti canonici gia' prodotti.
# Per i DUE DATASET NUOVI il default da impostare in training e' 256 (vedi sotto).
FAITH_N_SAMPLES = 128

# --- il numero da usare sui dataset nuovi, e perche' -------------------------------------
# Ricavato dalle CI gia' salvate nei 130 faithfulness.json, non a occhio:
#   a n=128 la CI half-width sul gain primario e' 0.133, contro una sd FRA RUN di 0.203
#   -> rumore = 0.66 x segnale. E non scende pooling i seed (vincolo 1: stesso campione).
#   n per portare CI_half a meta' della sd fra run = 221  ->  256 e' il numero tondo.
#   n=256: CI_half 0.094 (0.46x sd), costo 5 min SHAP + 4 min faith = ~9 min/run.
#   n=512: CI_half 0.066 (0.33x sd), ~18 min/run — rendimento decrescente, non vale.
FAITH_N_SAMPLES_TARGET_NEW_DATASETS = 256

# Tier di riprova/esplorazione: il validation intero (~800 righe) su POCHE run.
#   6 run (una per famiglia) x 800 campioni x 2.1 s = ~2.8 h, e risponde a "il numero cambia
#   se guardo tutto il validation invece di 256 righe?". Vedi anche xf.faith_subsets(k=5),
#   che confronta 5 batch DISGIUNTI da 128 (640/800 = 80% del validation).
FAITH_N_SAMPLES_EXPLORE = None    # None = non attivo; 800 per il giro largo

# Which runs the faithfulness loop processes (see the faithfulness cell for how they are used).
FAITH_ONLY_IMPL = None    # e.g. ["vanilla", "adaptive_gating"]; None = all discovered
FAITH_MAX_RUNS  = None    # e.g. 5 for a smoke test
FAITH_OVERWRITE = False   # False = reuse inline-computed faithfulness.json when present
FAITH_DRY_RUN   = False   # True = list what would run, load no models

# Small CSV summaries (no models loaded here).
cf_combined = pd.read_csv(os.path.join(cf_study_dir, "combined_summary.csv"))
ag_combined = pd.read_csv(os.path.join(ag_study_dir, "combined_summary.csv"))
print("params set | cf_combined", cf_combined.shape, "| ag_combined", ag_combined.shape)

# --- selezione delle run per il rigeneramento LORE (sezione "▶ Spiegazioni LORE") -------
#   LORE_DRY_RUN=True stampa cosa farebbe senza eseguire. Costo MISURATO: ~21 min per run.
#   LORE_PER_FAMILY=1 -> una run per famiglia (6 run, ~2,1 h). None -> tutte quelle selezionate.
#   LORE_ONLY_IMPL=["fixed_scaling"] + LORE_PER_FAMILY=None -> tutta la griglia alpha (45 run, ~16 h).
LORE_DRY_RUN    = False
LORE_ONLY_IMPL  = ["fixed_scaling", "learnable_scaling", "loss_regularization"]      #None
LORE_PER_FAMILY = 1
LORE_MAX_RUNS   = None   # <- head(n) applicato DOPO i filtri: con 3 run selezionate, 1 ne esegue UNA.
#   Lasciarlo a 1 e' il bug del 30-31/07: il test 3b (una run sola) non se ne accorge,
#   il 3c (tre run) ne esegue una e finisce in 15 min invece di un'ora. None = tutte.
#   LORE_ONLY_TAG filtra per setting_tag. Con ["a0p000","a1p000"] +
#   LORE_ONLY_IMPL=["fixed_scaling"] + LORE_PER_FAMILY=1 dà i due SOLI estremi
#   del test acido su alpha (2 run, ~42 min) invece dei 45 valori della griglia (~16 h).
LORE_ONLY_TAG   = ["a0p500", "init_a0p500", "lam1e-03"]       #None

# --- quanti campioni per LORE (xd.LORE_SA_KWARGS["max_explain_samples"], oggi 32) ---------
# A 28 s/campione LORE e' l'unica cosa davvero cara, e la potenza dice di NON alzarlo:
#   divario da risolvere            n/gruppo per p<0.05, potenza 80%      costo
#   a0p000 0.00 vs a0p500 0.25                 8                          4 min   <- gia' preso
#   benigni 0.05 vs maligni 0.54                6                          3 min   <- gia' preso
#   learnable 0.31 vs loss_reg 0.62            19                          9 min   <- gia' preso
#   vanilla 0.53 vs loss_reg 0.62             216                        102 min   <- fuori portata
#   fixed 0.25 vs learnable 0.31              411                        193 min   <- fuori portata
# Fra 32 e 128 campioni non si compra nessun confronto nuovo: i contrasti grandi sono gia'
# risolti a 32, quelli fra famiglie adiacenti restano irraggiungibili anche a 128 (4x il costo).
# DEFAULT: 32, e il budget va in PIU' RUN (5 seed x 6 famiglie = 30 run, ~7.5 h/dataset),
# perche' e' l'errore FRA SEED l'unico che oggi non e' misurato — ogni riga del memo LORE
# finisce con "da confermare su piu' seed".
# TIER DI ESPLORAZIONE: 128 campioni su 3 run (i due endpoint alfa + un centro) = 3 h.
LORE_N_SAMPLES_EXPLORE = 128


params set | cf_combined (5, 21) | ag_combined (5, 50)


## Setup — data, FeatureSpace, XAI cores

In [7]:
import pandas as pd

# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet').ffill()

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet').ffill()

In [8]:
print("Class Distribution Before Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

# Sampling
majority_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
minority_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([majority_df, minority_df], axis=0).sample(frac=1.0)

print("\nClass Distribution After Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

Class Distribution Before Sampling (%):
target
0    99.899858
1     0.100142
Name: proportion, dtype: float64

Class Distribution After Sampling (%):
target
0    66.611631
1    33.388369
Name: proportion, dtype: float64


In [9]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [10]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

# Num Train: 4360 | Num Valid: 1091


In [11]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [12]:
def build_augmenter():
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)
        image = tf.image.resize(image, size=target_size, method="area")
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, [*target_size, 3])
        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)
    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)

    # Decode before cache (so we don't decode every epoch)
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)

    if cache:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed, reshuffle_each_iteration=True)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        opt.experimental_slack = True
        ds = ds.with_options(opt)

    ds = ds.batch(batch_size, drop_remainder=drop_remainder)

    # Keep augmentation AFTER cache so it's still random per epoch
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTO)

    ds = ds.prefetch(AUTO)
    return ds

def _to_feature_dict(df):
    feat_dict = {}
    for c in df.columns:
        s = df[c]
        if s.dtype == object or str(s.dtype).startswith('string'):
            feat_dict[c] = s.astype(str).to_numpy()
        else:
            feat_dict[c] = s.to_numpy()
    return feat_dict

In [13]:
from tqdm import tqdm

# Build datasets
print("# Training:")
training_features = _to_feature_dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(
    training_ids, training_validation_hdf5, training_features, training_labels,
    batch_size=CFG.batch_size, shuffle=True, augment=True
)

print("# Validation:")
validation_features = _to_feature_dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(
    validation_ids, training_validation_hdf5, validation_features, validation_labels,
    batch_size=CFG.batch_size, shuffle=False, augment=False
)

print("# Testing:")
testing_features = _to_feature_dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(
    testing_ids, testing_hdf5, testing_features,
    batch_size=CFG.batch_size, shuffle=False, augment=False, cache=False
)

# Training:


I0000 00:00:1785630706.759758  934072 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22800 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5
Loading Images : 100%|██████████| 4360/4360 [00:01<00:00, 4042.95it/s]


# Validation:


Loading Images : 100%|██████████| 1091/1091 [00:00<00:00, 6401.53it/s]


# Testing:


Loading Images : 100%|██████████| 37576/37576 [00:05<00:00, 6422.82it/s]


In [14]:
# Evaluation set for the post-hoc metrics + validation-set identity guard.
EVAL_IS_CANONICAL = (EVAL_SET == "canonical")
if EVAL_IS_CANONICAL:
    _val_snap = os.path.join(BASE_PATH, "notebooks", "modality_attendance",
                             "validation_ids.SEED49.json")
    if os.path.exists(_val_snap):
        _snap = json.load(open(_val_snap))
        _match = (sorted(validation_df.isic_id.tolist()) == _snap["isic_ids_sorted"])
        # WARN (not raise): the shipped snapshot is a post-hoc regen, deterministic but NOT
        # verified against the runs' training split. Flip to `raise` once a real training run
        # has (re)written it (source='training_run').
        print(f"[EVAL GUARD] validation set vs snapshot (source={_snap.get('source','?')}): "
              f"{'MATCH' if _match else 'MISMATCH'} "
              f"(have {len(validation_df)}, snapshot {_snap['n']}).")
        if not _match:
            print("  \u26a0\ufe0f  reconstructed fold-0 set differs from the snapshot. If the snapshot came "
                  "from the real training run, post-hoc numbers will NOT correspond to the checkpoints.")
    else:
        print(f"[EVAL GUARD] no snapshot at {_val_snap}; run the training notebook once to write it.")
else:
    validation_df = df.query("fold in @EVAL_FOLDS").reset_index(drop=True)
    validation_ds = build_dataset(
        validation_df.isic_id.values, training_validation_hdf5,
        _to_feature_dict(validation_df[FEAT_COLS]), validation_df.target.values,
        batch_size=CFG.batch_size, shuffle=False, augment=False)
    print(f"[EVAL] custom set folds={EVAL_FOLDS}: {len(validation_df)} samples. "
          "SHAP tabular/joint faithfulness disabled; image faithfulness + on-the-fly diagnostics only.")


[EVAL GUARD] validation set vs snapshot (source=posthoc_regen_2026-07-28): MATCH (have 1091, snapshot 1091).


In [15]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

FORCE_READAPT_FEATURE_SPACE = False

_fs_file = os.path.join(FEATURE_SPACE_PATH, "feature_space.keras")

if os.path.exists(_fs_file) and not FORCE_READAPT_FEATURE_SPACE:
    feature_space = keras.saving.load_model(_fs_file)
    print(f"FeatureSpace LOADED from {_fs_file} (deterministic; not re-adapted)")
else:
    training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
    feature_space.adapt(training_ds_with_no_labels)
    os.makedirs(FEATURE_SPACE_PATH, exist_ok=True)
    feature_space.save(_fs_file)
    print(f"FeatureSpace ADAPTED and saved to {_fs_file}")

print(feature_space.get_encoded_features().shape)

FeatureSpace LOADED from /mnt/disk1/home/cgiovannoni/isic24-dum/artifacts/enhanced/feature_space/feature_space.keras (deterministic; not re-adapted)
(None, 127)


In [16]:
# FeatureSpace anti-drift guard: fail LOUD if the live FeatureSpace is not the canonical one
# the checkpoints were trained with (dim 127 / recorded fingerprint).
assert_feature_space_canonical(feature_space)

[FS GUARD] OK — live FeatureSpace matches canonical (dim 127, fp e8fa85b285bdcf62…, feature_space.MANIFEST.json)


(127, 'e8fa85b285bdcf62ee8540984c3e7ef8c4132011505e052a071301447cd86316')

In [17]:
# Apply Feature Processing
augment_fn = build_augmenter()

auto = tf.data.AUTOTUNE

training_ds = training_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
training_ds = training_ds.map(augment_fn, num_parallel_calls=auto)
training_ds = training_ds.prefetch(auto)

validation_ds = validation_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
validation_ds = validation_ds.prefetch(auto)

testing_ds = testing_ds.map(
    lambda x: {'images': x['images'], 'features': feature_space(x['features'])},
    num_parallel_calls=auto,
)
testing_ds = testing_ds.prefetch(auto)

In [18]:
# optimizer, loss & metric
adam = keras.optimizers.Adam(learning_rate=1e-4)
auc = keras.metrics.AUC(name='auc')
bce = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [19]:
_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

In [20]:
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1] == 128

per-feature dim sizes: {'sex': 3, 'anatom_site_general': 6, 'tbp_tile_type': 3, 'tbp_lv_location': 20, 'combined_anatomical_site': 23, 'attribution': 8, 'age_approx': 32, 'tbp_lv_nevi_confidence': 1, 'clin_size_long_diam_mm': 1, 'tbp_lv_areaMM2': 1, 'tbp_lv_area_perim_ratio': 1, 'tbp_lv_color_std_mean': 1, 'tbp_lv_deltaLBnorm': 1, 'tbp_lv_minorAxisMM': 1, 'lesion_size_ratio': 1, 'lesion_shape_index': 1, 'hue_contrast': 1, 'luminance_contrast': 1, 'lesion_color_difference': 1, 'border_complexity': 1, '3d_position_distance': 1, 'perimeter_to_area_ratio': 1, 'lesion_visibility_score': 1, 'symmetry_border_consistency': 1, 'color_consistency': 1, 'size_age_interaction': 1, 'hue_color_std_interaction': 1, 'lesion_severity_index': 1, 'shape_complexity_index': 1, 'color_contrast_index': 1, 'log_lesion_area': 1, 'mean_hue_difference': 1, 'std_dev_contrast': 1, 'color_shape_composite_index': 1, '3d_lesion_orientation': 1, 'overall_color_difference': 1, 'symmetry_perimeter_interaction': 1, 'compr

False

In [21]:
# ================================================================================
# XAI engines + faithfulness — imported, not defined here.
# --------------------------------------------------------------------------------
# Grad-CAM, the faithfulness machinery and the SHAP/LORE engines used to be ~1400 lines
# inline here AND byte-duplicated in the training notebook. They now live in
# `src/isic24/`, so both notebooks drive the SAME code.
#
# This also fixes a latent ordering bug: the faithfulness cell called
# `encoded_indices_for_raw` and `_finalize_plot`, which the notebook only defined further
# DOWN, in the archived appendix — running this notebook linearly raised NameError on the
# tabular method.
# ================================================================================
from isic24 import xai_faithfulness as xf
from isic24.xai_faithfulness import (
    explanation_faithfulness, faith_subsets, build_strata, persample_gains,
    stratified_table, pool_by_family, discover_curve_files, verify_reintegration,
    _bootstrap_ci, _build_faith_subset, _load_shap_values, _encode_raw, _score_logit,
)
from isic24.xai_driver import run_posthoc_xai, run_posthoc_xai_batch, make_xai_callbacks
from isic24.xai_engines import (
    MultiModalGradCAMEngine, MultiModalGradCAMCallback,
    encoded_indices_for_raw, aggregate_raw_importance, _finalize_plot, PLOT_SHOW,
)

# Hand the notebook's data objects to the faithfulness code once, plus the knobs from the
# PARAMETERS cell (they used to be read out of the caller's globals()).
xf.configure(
    xf.XaiContext(
        feat_cols=FEAT_COLS,
        training_df=training_df,
        validation_df=validation_df,
        validation_ds=validation_ds,
        feature_space=feature_space,
        to_feature_dict=_to_feature_dict,
        encoded_feature_names=encoded_feature_names,
    ),
    FAITH_N_SAMPLES=FAITH_N_SAMPLES,
    FAITH_BUDGETS=tuple(FAITH_BUDGETS),
    EVAL_IS_CANONICAL=EVAL_IS_CANONICAL,
)
print("[XAI] engines imported | eval set:", EVAL_SET, "| n_faith:", FAITH_N_SAMPLES)

# --------------------------------------------------------------------------------------
# CAMPIONE XAI — la scelta si fa QUI, una volta, e vale per SHAP, LORE e fedelta'.
#
# Campiona su LESIONI DISTINTE: `pos_sample = 5.0` con `replace=True` fa comparire la stessa
# lesione ~5 volte, e le copie sono repliche esatte (stessa immagine, stesse feature, stessa
# predizione). Spiegarle piu' volte restringe gli intervalli senza aggiungere informazione.
#
#   balance="pool"  prevalenza delle lesioni uniche (~0.09 su ISIC24) — livelli assoluti
#   balance="equal" meta' per classe — massima potenza sul contrasto fra classi, a parita'
#                   di costo; i confronti fra strategie restano validi (campione identico)
#   balance="all"   ogni lesione distinta, nessun campionamento (n ignorato)
#   balance=None    disarma tutto: torna alle prime N righe = i numeri gia' pubblicati
#
# Costo per run: fedelta' allineata 0.52 s/campione, LORE 28 s/campione.
# --------------------------------------------------------------------------------------
from isic24 import xai_subset as xs

XAI_SUBSET_BALANCE = "equal"    # None per riprodurre i numeri pubblicati
XAI_SUBSET_N       = 144        # = 72 positivi (TUTTI gli unici disponibili) + 72 negativi.
                                # 144 e' il massimo con balance="equal": oltre, i positivi
                                # unici finiscono. Usa tutta l'informazione che c'e'.
XAI_SUBSET_SEED    = 0

# I SETTING CENTRALI, uno per famiglia: il set di confronto di ogni tabella e figura.
# Serve esplicito perche' il master ha 45 righe di `fixed_scaling`, `a0p000` (ramo
# tabellare azzerato, tutti zeri) compresa: un `groupby("impl_name")` secco le mescola
# tutte nella media di famiglia e il numero che esce non vuol dire niente.
#
# Gli artefatti del campione vanno DENTRO la run, in `<run_dir>/xai/shap_tab/` e
# `<run_dir>/xai/lore_sa/` (`xs.out_dir_for`). Non esiste piu' una radice parallela.
CENTRAL_SETTINGS = {"vanilla": "baseline", "fixed_scaling": "a0p500",
                    "learnable_scaling": "init_a0p500",
                    "loss_regularization": "lam1e-03",
                    "controlled_fusion": "base", "adaptive_gating": "base"}

xai_subset_idx, xai_subset_info = xs.adopt(
    validation_df, n=XAI_SUBSET_N, balance=XAI_SUBSET_BALANCE, seed=XAI_SUBSET_SEED,
    snapshot_dir=os.path.join(BASE_PATH, "notebooks", "modality_attendance"))
XAI_SUBSET_FP = xai_subset_info["fingerprint"]


[XAI] engines imported | eval set: canonical | n_faith: 128
[campione] 144 lesioni DISTINTE (balance='equal', seed=0)
           per classe: {0: 72, 1: 72}   prevalenza 0.500
           uniche disponibili: {0: 727, 1: 73}
           verifica: 144/144 id distinti OK
           impronta: unique-equal-s0-n144-5671a4e59ff6998a
           attivo su: faithfulness, lore, shap
           snapshot: /mnt/disk1/home/cgiovannoni/isic24-dum/notebooks/modality_attendance/subset.unique-equal.n144.seed0.json
           ripristino del prefisso storico: xs.adopt(validation_df, balance=None)


In [22]:
# Post-hoc plotting + stale-path helpers.

def relmap(rd):
    # combined_summary.csv keeps the training machine's absolute paths; remap the '/runs/...'
    # suffix onto this machine's RUNS_PATH so checkpoints/curves resolve.
    if os.path.isdir(rd):
        return rd
    s = str(rd).replace("\\", "/"); i = s.find("/runs/")
    return os.path.join(os.path.dirname(RUNS_PATH), s[i + 1:]) if i >= 0 else rd


In [23]:
raise StopIteration

StopIteration: 

## Post-hoc diagnostics — gate faithfulness, calibration, gamma sweep

In [ ]:
# Generic AUC evaluator on a dataset
def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Build ablated inputs to remove/perturb one modality
def ablate_batch(x, mode, strength):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        x["images"] = (1.0 - strength) * img + strength * (tf.ones_like(img) * 0.5)  # gray image
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        mask = tf.random.uniform(tf.shape(feat)) >= strength
        x["features"] = tf.where(mask, feat, tf.zeros_like(feat))
    return x

# Evaluate AUC after modality ablation
def eval_auc_with_ablation(model, ds, mode, strength):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(ablate_batch(x, mode, strength), training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Simple Spearman (rank correlation) implementation
def spearman(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ra = a.argsort().argsort().astype(np.float32)
    rb = b.argsort().argsort().astype(np.float32)
    ra -= ra.mean()
    rb -= rb.mean()
    return float((ra @ rb) / (np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12))

# Convert probabilities to logits for impact comparison
def logit(p, eps=1e-6):
    p = tf.clip_by_value(tf.cast(p, tf.float32), eps, 1 - eps)
    return tf.math.log(p / (1 - p))

# Gate-faithfulness: do gate weights correlate with true modality impact?
def gate_faithfulness(model, ds, strength=1.0):
    gate_model = keras.Model(model.inputs, model.get_layer("gate_weights").output)

    wimg, wtab, dimg, dtab = [], [], [], []
    for x, _ in ds:
        p0 = tf.squeeze(model(x, training=False), -1)
        l0 = logit(p0)

        # Impact after image ablation
        pi = tf.squeeze(model(ablate_batch(x, "img", strength), training=False), -1)
        di = tf.abs(l0 - logit(pi))

        # Impact after tabular ablation
        pt = tf.squeeze(model(ablate_batch(x, "tab", strength), training=False), -1)
        dt = tf.abs(l0 - logit(pt))

        # Predicted gate weights
        w = gate_model(x, training=False)
        wimg.append(w[:, 0].numpy())
        wtab.append(w[:, 1].numpy())
        dimg.append(di.numpy())
        dtab.append(dt.numpy())

    wimg = np.concatenate(wimg)
    wtab = np.concatenate(wtab)
    dimg = np.concatenate(dimg)
    dtab = np.concatenate(dtab)

    return {
        "spearman_w_img_delta_img": spearman(wimg, dimg),
        "spearman_w_tab_delta_tab": spearman(wtab, dtab),
    }

# Calibration metrics: Brier score + ECE
def brier_ece(model, ds, n_bins=15):
    y_all, p_all = [], []
    for x, y in ds:
        p = tf.squeeze(model(x, training=False), -1)
        y_all.append(tf.cast(tf.squeeze(y), tf.float32).numpy())
        p_all.append(tf.cast(p, tf.float32).numpy())

    y = np.concatenate(y_all)
    p = np.concatenate(p_all)
    brier = float(np.mean((p - y) ** 2))

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(p)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.any(m):
            conf = float(np.mean(p[m]))
            acc = float(np.mean(y[m]))
            ece += (np.sum(m) / N) * abs(acc - conf)

    return {"brier": brier, "ece": float(ece)}

In [ ]:
# post-training diagnostics

all_diag = []
for _, row in ag_combined.iterrows():
    ag_run_dir = row["run_dir"]
    # combined_summary.csv stores June paths (/home/claudio/...); remap the
    # "/runs/..." suffix onto this machine's RUNS_PATH so checkpoints load.
    if not os.path.isdir(ag_run_dir):
        _s = str(ag_run_dir).replace("\\", "/")
        _i = _s.find("/runs/")
        if _i >= 0:
            ag_run_dir = os.path.join(os.path.dirname(RUNS_PATH), _s[_i + 1:])
    best_ckpt = os.path.join(ag_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    ag_model = keras.models.load_model(best_ckpt, compile=False, safe_mode=False)
    assert_fs_matches_model(ag_model, feature_space, row.get("run_id"))

    gate_model = keras.Model(ag_model.inputs, ag_model.get_layer("gate_weights").output)
    gx, _ = next(iter(validation_ds))
    gw = gate_model(gx, training=False).numpy()

    base_auc = eval_auc(ag_model, validation_ds)
    img_auc = eval_auc_with_ablation(ag_model, validation_ds, "img", 1.0)
    tab_auc = eval_auc_with_ablation(ag_model, validation_ds, "tab", 1.0)

    diag = {
        "run_id": row["run_id"],
        "setting_tag": row["setting_tag"],
        "seed": int(row["seed"]),
        "entropy_weight": float(row["entropy_weight"]),
        "run_dir": ag_run_dir,
        "gate_weight_mean_img": float(np.mean(gw[:, 0])),
        "gate_weight_mean_tab": float(np.mean(gw[:, 1])),
        "base_auc": base_auc,
        "img_drop_auc": base_auc - img_auc,
        "tab_drop_auc": base_auc - tab_auc,
        **gate_faithfulness(ag_model, validation_ds, strength=1.0),
        **brier_ece(ag_model, validation_ds),
    }

    os.makedirs(os.path.join(ag_run_dir, "diagnostics"), exist_ok=True)
    with open(os.path.join(ag_run_dir, "diagnostics", "adaptive_gating_diagnostics.json"), "w") as f:
        json.dump(diag, f, indent=2)
    all_diag.append(diag)

    # ---- diagnostic plots (per run), saved to the run's plots/ folder ----
    plots_dir = os.path.join(ag_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # gather full-validation gate weights, labels and predictions in one pass
    wimg_all, wtab_all, y_true_all, p_pred_all = [], [], [], []
    for xb, yb in validation_ds:
        wb = gate_model(xb, training=False).numpy()
        pb = tf.squeeze(ag_model(xb, training=False), -1).numpy()
        wimg_all.append(wb[:, 0]); wtab_all.append(wb[:, 1])
        y_true_all.append(np.asarray(tf.squeeze(yb)).reshape(-1))
        p_pred_all.append(np.asarray(pb).reshape(-1))
    wimg_all = np.concatenate(wimg_all); wtab_all = np.concatenate(wtab_all)
    y_true_all = np.concatenate(y_true_all); p_pred_all = np.concatenate(p_pred_all)

    # 1) gate-weight distribution
    plt.figure(figsize=(7, 4))
    plt.hist(wimg_all, bins=30, alpha=0.6, label="w_img")
    plt.hist(wtab_all, bins=30, alpha=0.6, label="w_tab")
    plt.xlabel("gate weight"); plt.ylabel("count")
    plt.title(f"Adaptive gating: gate-weight distribution (seed {int(row['seed'])})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "gate_weight_distribution.png")

    # 2) modality ablation: base AUC and AUC drop after ablating each modality
    plt.figure(figsize=(7, 4))
    labels = ["base_auc", "img_drop_auc", "tab_drop_auc"]
    vals = [diag["base_auc"], diag["img_drop_auc"], diag["tab_drop_auc"]]
    plt.bar(labels, vals, color=["#4c72b0", "#dd8452", "#55a868"])
    for j, v in enumerate(vals):
        plt.text(j, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.ylabel("AUC"); plt.grid(alpha=0.3, axis="y")
    plt.title(f"Adaptive gating: modality ablation (seed {int(row['seed'])})")
    _finalize_plot(plots_dir, "modality_ablation_auc_drop.png")

    # 3) calibration reliability diagram
    n_bins = 15
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_conf, bin_obs = [], []
    for j in range(n_bins):
        lo, hi = edges[j], edges[j + 1]
        m = (p_pred_all >= lo) & (p_pred_all < hi) if j < n_bins - 1 else (p_pred_all >= lo) & (p_pred_all <= hi)
        if np.any(m):
            bin_conf.append(float(np.mean(p_pred_all[m])))
            bin_obs.append(float(np.mean(y_true_all[m])))
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
    plt.plot(bin_conf, bin_obs, marker="o", label="model")
    plt.xlabel("mean predicted probability"); plt.ylabel("observed frequency")
    plt.title(f"Calibration (Brier={diag['brier']:.3f}, ECE={diag['ece']:.3f})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "calibration_reliability.png")

    keras.backend.clear_session()
    gc.collect()

ag_diag_df = pd.DataFrame(all_diag)
ag_diag_df.to_csv(os.path.join(ag_study_dir, "adaptive_gating_diagnostics.csv"), index=False)
display(ag_diag_df)

In [ ]:
# Post-hoc gamma sweep over every controlled-fusion run
# gamma_tab > 1 enhances tabular / penalizes image (gamma_img < 1)
# gamma_tab < 1 penalizes tabular / enhances image (gamma_img > 1)
DELTAS = np.round(np.arange(-0.5, 0.501, 0.125), 3)
POSTHOC_SETTINGS = []
for d in DELTAS:
    POSTHOC_SETTINGS.append({
        "tag": f"d{d:+.3f}".replace(".", "p").replace("+", "plus").replace("-", "minus"),
        "delta": float(d),
        "gamma_img": float(1.0 - d),
        "gamma_tab": float(1.0 + d),
        "mode": "tab_enhance_img_penalize" if d > 0 else ("tab_penalize_img_enhance" if d < 0 else "neutral"),
    })

posthoc_rows = []
for _, row in cf_combined.iterrows():
    cf_run_dir = row["run_dir"]
    ckpt_path = os.path.join(cf_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
    assert_fs_matches_model(best_model, feature_space, row.get("run_id"))
    fusion_layer = best_model.get_layer("fusion_with_gamma")

    run_rows = []
    for s in POSTHOC_SETTINGS:
        fusion_layer.gamma_img.assign(s["gamma_img"])
        fusion_layer.gamma_tab.assign(s["gamma_tab"])
        auc_val = eval_auc(best_model, validation_ds)

        rec = {
            "run_id": row["run_id"],
            "setting_tag": row["setting_tag"],
            "seed": int(row["seed"]),
            "run_dir": cf_run_dir,
            "posthoc_id": s["tag"],
            "delta": s["delta"],
            "gamma_img": s["gamma_img"],
            "gamma_tab": s["gamma_tab"],
            "mode": s["mode"],
            "val_auc": auc_val,
            "base_model_name": row["model_name"],
        }
        run_rows.append(rec)
        posthoc_rows.append(rec)

    posthoc_df = pd.DataFrame(run_rows).sort_values("gamma_tab").reset_index(drop=True)
    diagnostics_dir = os.path.join(cf_run_dir, "diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    posthoc_df.to_csv(os.path.join(diagnostics_dir, "posthoc_gamma_summary.csv"), index=False)

    plots_dir = os.path.join(cf_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 4))
    plt.plot(posthoc_df["gamma_tab"], posthoc_df["val_auc"], marker="o")
    plt.axvline(1.0, linestyle="--", linewidth=1)
    plt.xlabel("gamma_tab")
    plt.ylabel("val_auc")
    plt.title(f"Controlled fusion post-hoc: val_auc vs gamma_tab (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_val_auc_vs_gamma_tab.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.scatter(posthoc_df["gamma_img"], posthoc_df["gamma_tab"], c=posthoc_df["val_auc"])
    for _, r in posthoc_df.iterrows():
        plt.text(r["gamma_img"], r["gamma_tab"], f"{r['val_auc']:.3f}", fontsize=8)
    plt.xlabel("gamma_img")
    plt.ylabel("gamma_tab")
    plt.title(f"Controlled fusion post-hoc sweep (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_gamma_sweep_scatter.png"), dpi=300)
    plt.close()

    keras.backend.clear_session()
    gc.collect()

posthoc_all_df = pd.DataFrame(posthoc_rows).sort_values(["seed", "gamma_tab"]).reset_index(drop=True)
posthoc_all_df.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_summary.csv"), index=False)

posthoc_agg = (
    posthoc_all_df.groupby(["delta", "gamma_img", "gamma_tab"], dropna=False)["val_auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
posthoc_agg.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_aggregated.csv"), index=False)
display(posthoc_all_df.head(20))
display(posthoc_agg)

## Explanation Faithfulness (Deletion / Insertion AUC)

`faithfulness_gain = deletion_auc_explanation - deletion_auc_random` (piu alto = meglio), calcolata per `tabular` (SHAP), `image` (Grad-CAM) e `joint`. Riusa gli array SHAP gia salvati (nessun re-fit del GradientExplainer). Le tre celle seguenti definiscono il core, lo eseguono post-hoc su tutti i run gia allenati e aggregano i risultati.

In [ ]:
# ---------------------------------------------------------------------
# Post-hoc faithfulness over ALL trained runs (no retraining).
# Discovers run dirs from single_run_summary.csv, loads each best_*.keras
# checkpoint, computes deletion/insertion faithfulness and saves per-run
# artifacts + a combined CSV.
# ---------------------------------------------------------------------
FAITH_RUN_GLOB = os.path.join(RUNS_PATH, "**", "single_run_summary.csv")  # any depth: 3-level single + 4-level multi-seed
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
os.makedirs(FAITH_OUT_DIR, exist_ok=True)
_faith_name = "faithfulness" if globals().get("EVAL_IS_CANONICAL", True) else f"faithfulness_{EVAL_SET}"
_faith_methods = None if globals().get("EVAL_IS_CANONICAL", True) else ("image",)

# FAITH_ONLY_IMPL / FAITH_OVERWRITE / FAITH_MAX_RUNS / FAITH_DRY_RUN come from the PARAMETERS cell.

_SETTING_PARAM_COLS = ("alpha", "init_alpha", "lambda_reg",
                       "gamma_img", "gamma_tab", "entropy_weight")

summ_paths = sorted(glob.glob(FAITH_RUN_GLOB, recursive=True))

selected_runs = []
n_reuse = 0
n_compute = 0
for sp in summ_paths:
    try:
        row = pd.read_csv(sp).iloc[0].to_dict()
    except Exception as e:
        print("  skip (bad csv):", sp, e)
        continue

    impl = row.get("impl_name")
    if FAITH_ONLY_IMPL and impl not in FAITH_ONLY_IMPL:
        continue

    run_dir = row.get("run_dir")
    if not isinstance(run_dir, str) or not os.path.isdir(run_dir):
        run_dir = os.path.dirname(sp)
    model_name = row.get("model_name")
    best_epoch = int(row.get("best_epoch", -1))
    ckpt = os.path.join(run_dir, "checkpoints", f"best_{model_name}.keras")
    if not os.path.exists(ckpt) or best_epoch < 0:
        print("  skip (no ckpt / best_epoch):", run_dir)
        continue

    fjson = os.path.join(run_dir, "xai", "faithfulness", f"{_faith_name}.json")
    will_reuse = os.path.exists(fjson) and not FAITH_OVERWRITE
    n_reuse += int(will_reuse)
    n_compute += int(not will_reuse)
    selected_runs.append((sp, row, run_dir, model_name, best_epoch, ckpt, fjson))

if FAITH_MAX_RUNS is not None:
    selected_runs = selected_runs[:int(FAITH_MAX_RUNS)]
    n_reuse = sum(os.path.exists(r[6]) and not FAITH_OVERWRITE for r in selected_runs)
    n_compute = len(selected_runs) - n_reuse

print(f"[FAITH] discovered {len(summ_paths)} run summaries | selected={len(selected_runs)} "
      f"| reuse={n_reuse} | compute={n_compute}")
if FAITH_DRY_RUN:
    display(pd.DataFrame([{"impl_name": r[1].get("impl_name"),
                           "setting_tag": r[1].get("setting_tag"),
                           "seed": r[1].get("seed"),
                           "run_dir": r[2],
                           "action": "reuse" if os.path.exists(r[6]) and not FAITH_OVERWRITE else "compute"}
                          for r in selected_runs]).head(50))
    raise SystemExit("FAITH_DRY_RUN=True: no models loaded.")

faith_rows = []
for sp, row, run_dir, model_name, best_epoch, ckpt, fjson in selected_runs:
    impl = row.get("impl_name")
    if os.path.exists(fjson) and not FAITH_OVERWRITE:
        res = json.load(open(fjson))
        res = {k: v for k, v in res.items() if k not in ("curves", "q_grid")}
    else:
        t0 = time.time()
        try:
            keras.backend.clear_session()
            model = keras.models.load_model(ckpt, compile=False, safe_mode=False)
            assert_fs_matches_model(model, feature_space, run_dir)
            res = explanation_faithfulness(model, run_dir, best_epoch, save=True, verbose=False,
                                           out_name=_faith_name, methods=_faith_methods)
        except Exception as e:
            print(f"  [WARN] {impl}/{row.get('setting_tag')}: {type(e).__name__}: {e}")
            keras.backend.clear_session(); gc.collect()
            continue
        keras.backend.clear_session(); gc.collect()
        res["faith_seconds"] = round(time.time() - t0, 2)

    rec = {
        "impl_name": impl,
        "setting_tag": row.get("setting_tag"),
        "seed": row.get("seed"),
        "best_val_auc": row.get("best_val_auc"),
        "run_dir": run_dir,
        **{c: row[c] for c in _SETTING_PARAM_COLS if c in row},
        **{k: v for k, v in res.items() if not isinstance(v, (list, dict))},
    }
    faith_rows.append(rec)
    print(f"  ok {impl:>18}/{str(row.get('setting_tag')):<14} "
          f"gain={res.get('faithfulness_gain', float('nan')):+.4f} "
          f"({res.get('faith_seconds', '?')}s)")

faith_df = pd.DataFrame(faith_rows)
out_csv = os.path.join(FAITH_OUT_DIR, f"combined_{_faith_name}.csv")
# MERGE (not overwrite): keep prior rows for runs not recomputed in this pass,
# replace rows for run_dirs processed now. Safe with FAITH_ONLY_IMPL filtering.
if os.path.exists(out_csv) and len(faith_df):
    _prev = pd.read_csv(out_csv)
    _prev = _prev[~_prev["run_dir"].isin(faith_df["run_dir"])]
    faith_df = pd.concat([_prev, faith_df], ignore_index=True)
elif os.path.exists(out_csv) and not len(faith_df):
    faith_df = pd.read_csv(out_csv)
faith_df.to_csv(out_csv, index=False)
print(f"[FAITH] saved {out_csv}  shape={faith_df.shape}")
display(faith_df.head(20))


In [ ]:
# ---------------------------------------------------------------------
# Cross-strategy aggregation: mean/std over seeds, bootstrap CI,
# comparison plots. Run after the post-hoc loop above.
# ---------------------------------------------------------------------
faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
fdf = pd.read_csv(faith_csv)

# --- keep ONLY canonical runs: drop seed_49 smoke runs and the superseded
#     duplicate controlled_fusion run so nothing is double-counted here.
#     (CANONICAL is defined in the "canonical study dirs" cell above.)
_canon_ts = {impl: os.path.basename(p) for impl, p in CANONICAL.items()}
fdf["_ts"] = fdf["run_dir"].astype(str).str.extract(r"/(\d{8}-\d{6})/")[0]
fdf = fdf[fdf.apply(lambda r: _canon_ts.get(r["impl_name"]) == r["_ts"], axis=1)].copy()
fdf = fdf.drop(columns=["_ts"]).reset_index(drop=True)
print(f"[FAITH] aggregating over {len(fdf)} canonical runs: "
      f"{fdf['impl_name'].value_counts().to_dict()}")
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
FAITH_SHOW_AGG_PLOTS = False  # save aggregate PNGs without flooding notebook output

gain_cols = [c for c in fdf.columns
             if c.startswith("faithfulness_") and c.endswith("_gain")]
val_cols = [c for c in (gain_cols + ["faithfulness_gain", "best_val_auc"]) if c in fdf.columns]

# ---- mean / std / n per (impl_name, setting_tag) over seeds ----
agg = (fdf.groupby(["impl_name", "setting_tag"], dropna=False)[val_cols]
          .agg(["mean", "std", "count"]))
agg.to_csv(os.path.join(FAITH_OUT_DIR, "faithfulness_summary.csv"))
print("[FAITH] per-setting summary:")
display(agg)

# ---- bootstrap CI of primary faithfulness_gain per strategy (resample runs) ----
ci_rows = []
for impl, sub in fdf.groupby("impl_name"):
    m, lo, hi = _bootstrap_ci(sub["faithfulness_gain"].values)
    ci_rows.append({"impl_name": impl, "mean_gain": m, "ci_lo": lo, "ci_hi": hi, "n_runs": len(sub)})
ci_df = pd.DataFrame(ci_rows).sort_values("mean_gain", ascending=False).reset_index(drop=True)
ci_df.to_csv(os.path.join(FAITH_OUT_DIR, "faithfulness_gain_ci.csv"), index=False)
print("[FAITH] strategy ranking (primary deletion gain, bootstrap 95% CI):")
display(ci_df)

# ---- bar chart: faithfulness_gain per strategy with CI ----
plt.figure(figsize=(9, 5))
x = np.arange(len(ci_df))
yerr = [ci_df["mean_gain"] - ci_df["ci_lo"], ci_df["ci_hi"] - ci_df["mean_gain"]]
plt.bar(x, ci_df["mean_gain"], yerr=yerr, capsize=4, color="#4c72b0", edgecolor="black")
plt.axhline(0.0, color="gray", lw=1)
plt.xticks(x, ci_df["impl_name"], rotation=30, ha="right")
plt.ylabel("faithfulness_gain (primary deletion)")
plt.title("Explanation faithfulness gain per strategy (bootstrap 95% CI)")
_finalize_plot(FAITH_OUT_DIR, "faithfulness_gain_by_strategy.png", show=FAITH_SHOW_AGG_PLOTS)

# ---- deletion-curve overlay per strategy (mean across runs) ----
def _overlay_deletion(method):
    plt.figure(figsize=(9, 5))
    plotted = False
    for impl, sub in fdf.groupby("impl_name"):
        cs, qg = [], None
        for _, r in sub.iterrows():
            npz = os.path.join(str(r["run_dir"]), "xai", "faithfulness", "faithfulness_curves.npz")
            if not os.path.exists(npz):
                continue
            d = np.load(npz)
            key = f"del_exp_{method}"
            if key not in d.files:
                continue
            cs.append(d[key].mean(axis=0)); qg = d["q_grid"]
        if cs:
            plt.plot(qg, np.mean(cs, axis=0), marker="o", label=impl); plotted = True
    if plotted:
        plt.xlabel("budget q removed"); plt.ylabel("mean logit drop C_exp(q)")
        plt.title(f"Explanation-guided deletion curves ({method}) per strategy")
        plt.legend(); plt.grid(alpha=0.3)
        _finalize_plot(FAITH_OUT_DIR, f"deletion_curves_{method}_by_strategy.png", show=FAITH_SHOW_AGG_PLOTS)
    else:
        plt.close()

for _m in ("joint", "tabular", "image"):
    _overlay_deletion(_m)

print("[FAITH] aggregation done ->", FAITH_OUT_DIR)


In [ ]:
# ---------------------------------------------------------------------
# MASTER RESULTS TABLE -> artifacts/enhanced/summaries/
# One row per run + a (family,setting) mean/std/count summary. Paper Table 1
# source + AUC-vs-faithfulness trade-off scatter source.
# Requires: canonical loader cell + regenerated combined_faithfulness.csv.
# ---------------------------------------------------------------------
os.makedirs(SUMMARIES_PATH, exist_ok=True)
_frames = []
for _impl, _d in CANONICAL.items():
    _p = os.path.join(_d, "combined_summary.csv")
    if os.path.exists(_p):
        _df = pd.read_csv(_p); _df["impl_name"] = _impl; _frames.append(_df)
master = pd.concat(_frames, ignore_index=True)
_faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
if os.path.exists(_faith_csv):
    _fdf = pd.read_csv(_faith_csv)
    _keep = [c for c in ["run_dir","faithfulness_gain","faithfulness_tabular_del_gain",
             "faithfulness_image_del_gain"] if c in _fdf.columns]
    _fdf = _fdf[_keep].copy()
    # run_dir absolute prefixes differ across machines (June: /home/claudio/...,
    # now: /mnt/disk1/home/cgiovannoni/...). combined_summary.csv keeps the OLD
    # prefix, combined_faithfulness.csv the NEW one, so a raw run_dir join is
    # all-NaN. Join on the machine-independent suffix starting at "/runs/".
    def _relkey(p):
        p = str(p).replace("\\", "/")
        i = p.find("/runs/")
        return p[i:] if i >= 0 else p
    master["_key"] = master["run_dir"].map(_relkey)
    _fdf["_key"] = _fdf["run_dir"].map(_relkey)
    _fdf = _fdf.drop(columns=["run_dir"]).drop_duplicates("_key")
    _dupe = [c for c in _fdf.columns if c != "_key" and c in master.columns]
    master = master.drop(columns=_dupe).merge(_fdf, on="_key", how="left").drop(columns=["_key"])

# LORE fidelity: combined_summary.csv only carries it for families whose runner
# logged it inline. Fill vanilla + controlled_fusion from the post-hoc CSV and
# loss_regularization from each run's per-epoch CSV (last epoch, context mode --
# same semantics as the inline lore_fidelity).
import glob as _g
def _lkey(p):
    p = str(p).replace("\\", "/")
    i = p.find("/runs/")
    return p[i:] if i >= 0 else p
_lore_map = {}
_lp = os.path.join(RUNS_PATH, "_faithfulness", "lore_fidelity_posthoc.csv")
if os.path.exists(_lp):
    for _, _r in pd.read_csv(_lp).iterrows():
        _lore_map[_lkey(_r["run_dir"])] = float(_r["lore_fidelity"])
for _pe in _g.glob(os.path.join(CANONICAL["loss_regularization"], "**", "lore_fidelity_per_epoch.csv"), recursive=True):
    _e = pd.read_csv(_pe)
    if len(_e):
        _lore_map[_lkey(os.path.dirname(os.path.dirname(os.path.dirname(_pe))))] = float(_e.iloc[-1]["lore_fidelity_context"])
if _lore_map and "lore_fidelity" in master.columns:
    master["lore_fidelity"] = master["lore_fidelity"].fillna(master["run_dir"].map(_lkey).map(_lore_map))
_metrics = [c for c in ["best_val_auc","brier","ece","modality_balance","lore_fidelity",
            "faithfulness_gain","faithfulness_tabular_del_gain","faithfulness_image_del_gain"]
            if c in master.columns]
_group = [c for c in ["impl_name","setting_tag"] if c in master.columns]
summary = master.groupby(_group, dropna=False)[_metrics].agg(["mean","std","count"])
master.to_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"), index=False)
summary.to_csv(os.path.join(SUMMARIES_PATH, "master_results_summary.csv"))
print("[SUMMARY] wrote master_results_rows.csv + master_results_summary.csv ->", SUMMARIES_PATH)
display(summary)


## Low-budget faithfulness diagnosis (deletion ↔ insertion agreement)

Re-integrates the **saved** per-q curves at each `Q_MAX_REPORT` cutoff — no model calls. At the clean cutoff (`q≤0.20`) explanation-guided deletion and insertion **agree**; the divergence appears only at high budget (`q≤0.60`) — the OOD artifact (`FINDINGS_insertion_deletion_2026-07-21.md`).

> On a custom `EVAL_SET` this cell still reads the canonical `faithfulness_curves.npz` (the canonical-narrative diagnostic); it will not reflect the custom-eval numbers computed above.

In [ ]:
# Accordo deletion <-> insertion, basso vs pieno budget — dalle curve GIA' SALVATE (no GPU).
#
# CORREZIONE 2026-07-30 — questa cella pescava i run da `combined_faithfulness.csv`, che ha
# 130 righe: i 110 run canonici PIU' 20 study dir superate e seed_49. Mediare anche quei 20
# sposta le medie di famiglia (controlled_fusion +0.070 invece di +0.149). Ora passa da
# `discover_curve_files`, che filtra sul registro canonico `summaries/master_results_rows.csv`.
# I numeri stampati da questa cella PRIMA di questa data non sono confrontabili con questi.
# --- autosufficiente: questa sezione NON richiede il setup dati/FeatureSpace ---
# Servono solo le celle 1-7 (path + PARAMETERS). Tutto qui sotto e' aritmetica sulle curve
# gia' salvate: nessun modello, nessun dataset, nessuna GPU. L'import e' idempotente se la
# cella "XAI engines" e' gia' stata eseguita.
from isic24 import xai_faithfulness as xf

_files_canon = xf.discover_curve_files(ART_PATH)
_files_canon = _files_canon[_files_canon.source == "canonical"]
print(f"[curve] {_files_canon.run_id.nunique()} run canonici (il vecchio glob ne prendeva 130)")

_agree = xf.stratified_table(ART_PATH, files=_files_canon, q_maxes=tuple(Q_MAX_REPORT),
                             verbose=False)
_agree = _agree[(_agree.stratum == "all") & (_agree.method == "joint")]

# Una riga per (cutoff, run): la correlazione si misura fra run, la tabella si mostra per famiglia.
_ps = (_agree.pivot_table(index=["q_max", "impl_name", "run_id"], columns="direction",
                          values="gain").reset_index())
_corr = _ps.groupby("q_max")[["del", "ins"]].apply(lambda s: s["del"].corr(s["ins"]))
print("\ncorrelazione deletion<->insertion fra run, per cutoff:")
print(_corr.to_string())

# NB media NON pesata sui run, e su `gain` GREZZA: e' la diagnosi storica di questa
# cella. La tabella di sintesi piu' sotto usa `pool_by_family` (pesata per numero di
# campioni) e `gain_norm`. I due numeri non coincidono, e non devono.
fam = _ps.groupby(["q_max", "impl_name"])[["del", "ins"]].mean()
fam.columns = ["deletion_gain", "insertion_gain"]
fam["sign_agree"] = np.sign(fam["deletion_gain"]) == np.sign(fam["insertion_gain"])

_out_csv = os.path.join(RUNS_PATH, "_faithfulness", "low_budget_faithfulness.csv")
fam.reset_index().to_csv(_out_csv, index=False)
print("scritto", _out_csv)
display(fam)

## ▶ Risultati finali — tutte le tabelle di fedeltà, corrette

Questa sezione **rigenera da zero** ogni tabella di fedeltà partendo dalle curve per campione
già su disco. Nessun modello caricato, nessuna GPU, ~1–2 minuti: è pura re-integrazione.

Copre in un colpo solo quello che prima era sparso fra script one-off e tabelle parziali:

| sorgente | attributore | metodi | direzioni | budget |
| --- | --- | --- | --- | --- |
| `canonical` | SHAP **GradientExplainer** | tabular, image, joint | deletion + insertion | tutti i `Q_MAX_REPORT` |
| `aligned` | **Permutation** allineato (logit) | tabular | deletion + insertion | tutti i `Q_MAX_REPORT` |
| `aligned_signed` | Permutation, per **convenzione di segno** | tabular | deletion + insertion | tutti i `Q_MAX_REPORT` |

e ogni cella è stratificata per **classe** e per **confidenza × correttezza** (le quattro
casistiche sicuro/incerto × giusto/sbagliato).

> **Due colonne, non una.** `gain` è l'integrale grezzo, confrontabile con tutto ciò che è
> stato pubblicato. `gain_norm = gain / q_width` è l'unica confrontabile **fra budget diversi**:
> un'integrale su `q≤0.2` vale meccanicamente ~⅓ di uno su `q≤0.6` prima ancora che entri in
> gioco la qualità della spiegazione.

In [ ]:
# @@STRATA_SECTION@@ 1/3 — costruzione
# ================================================================================
# Tabella di fedeltà completa, rigenerata dalle curve su disco. Zero GPU.
# Sostituisce: gli script one-off `scripts/xai_2026-07-28/run_strata_signed.py` e le
# tabelle parziali citate nei report (che mostravano solo deletion a budget pieno).
# ================================================================================
# --- autosufficiente: questa sezione NON richiede il setup dati/FeatureSpace ---
# Servono solo le celle 1-7 (path + PARAMETERS). Tutto qui sotto e' aritmetica sulle curve
# gia' salvate: nessun modello, nessun dataset, nessuna GPU. L'import e' idempotente se la
# cella "XAI engines" e' gia' stata eseguita.
from isic24 import xai_faithfulness as xf

FAITH_STRATA_SIGNED = True    # includi le varianti con segno (oggi: 6 run, 1 per famiglia)

_files = xf.discover_curve_files(ART_PATH, include_signed=FAITH_STRATA_SIGNED)
print("[curve] copertura per sorgente:")
display(_files.groupby("source").agg(run=("run_id", "nunique"), file=("npz_path", "size")))

# GUARD — la media dei gain per campione deve riprodurre l'aggregato salvato nei JSON.
# Se questo non passa, ogni numero sotto è di un'altra grandezza e va fermato tutto.
_probs = xf.verify_reintegration(_files[_files.source == "canonical"].npz_path.iloc[0])
print("[guard] verify_reintegration:", "OK (<1e-9)" if not _probs else f"** PROBLEMI ** {_probs}")

strata = xf.stratified_table(ART_PATH, files=_files, q_maxes=tuple(Q_MAX_REPORT), verbose=False)
pooled = xf.pool_by_family(strata)

_p1 = os.path.join(RUNS_PATH, "_faithfulness", "faithfulness_strata.csv")
_p2 = os.path.join(RUNS_PATH, "_faithfulness", "faithfulness_strata_pooled.csv")
strata.to_csv(_p1, index=False)
pooled.to_csv(_p2, index=False)
print(f"\n[scritto] {_p1}   ({len(strata)} righe, una per run x sorgente x cella)")
print(f"[scritto] {_p2}   ({len(pooled)} righe, aggregate per famiglia)")
print("\nEntrambi sono DERIVATI: si rigenerano da questa cella in qualunque momento, "
      "non sono artefatti canonici.")

In [ ]:
# @@STRATA_SECTION@@ 2/3 — la tabella di sintesi
# Una riga per famiglia, tutti gli attributori affiancati. Deletion, stratum "all".
# La colonna mostrata e' `gain_norm` (per unita' di budget): e' l'unica che si puo'
# leggere attraverso cutoff diversi. Positivo = spiegazione fedele.
_h = pooled[(pooled.stratum == "all") & (pooled.direction == "del")].copy()
_h["attributore"] = np.where(
    _h.source == "canonical", "SHAPgrad:" + _h.method,
    np.where(_h.source == "aligned", "perm:tabular", "signed:" + _h["mode"]))

_piv = _h.pivot_table(index="impl_name", columns=["attributore", "q_max"], values="gain_norm")
print("gain_norm — deletion, tutti i campioni (positivo = fedele)")
display(_piv.round(3))

# Quante run stanno dietro ogni numero: e' la differenza fra un risultato e un aneddoto.
print("\nrun per cella (n_runs):")
display(_h.pivot_table(index="impl_name", columns="attributore", values="n_runs", aggfunc="max"))

print("""
LEGGERE COSI':
  SHAPgrad:*   = SHAP GradientExplainer, il ramo indicato (tabular/image/joint).
  perm:tabular = attributore Permutation allineato al logit, ranking |attribution|.
  signed:*     = lo stesso attributore, ma separando evidenza PRO e CONTRO la classe.
                 `target` e' la variante condizionata alla classe vera: e' quella da citare.

  `perm:tabular` e `signed:abs` sono LO STESSO attributore con LO STESSO ranking |phi|:
  se differiscono (vanilla -0.094 vs -0.041) e' perche' il primo media 5-45 run e il
  secondo ne ha uno solo. E' una differenza di popolazione, non di metodo.

CAVEAT CHE VIAGGIANO CON LA TABELLA:
  * le colonne `signed:*` hanno n_runs=1 (6 run, uno per famiglia, scelti per comodita'):
    il confronto FRA famiglie sotto il protocollo signed non e' ancora difendibile;
  * `fixed_scaling` sotto `signed:*` e' rappresentata dal solo endpoint immagine-only
    (alpha=0), dove ogni numero con segno vale 0.0 per costruzione: non e' una misura;
  * stessa finestra di budget != stessa unita' di evidenza (a q=0.05 il ramo tabellare
    toglie 2 feature su 39, quello immagine 13 patch su 256, con sostituzioni diverse).
""")

In [ ]:
# @@STRATA_SECTION@@ 3/3 — per classe e per confidenza x correttezza
# Le quattro casistiche (sicuro|incerto) x (giusto|sbagliato), piu' le due di classe.
# Al cutoff basso, che e' il regime pulito: a budget pieno la metrica e' dominata dall'OOD.
STRATA_SHOW = ["all", "class_pos", "class_neg",
               "sure_correct", "sure_wrong", "unsure_correct", "unsure_wrong"]
_QLOW = min(Q_MAX_REPORT)

_s = pooled[(pooled.direction == "del") & (pooled.q_max == _QLOW)
            & (pooled.stratum.isin(STRATA_SHOW))]

for _src, _mode, _meth, _title in [
        ("aligned", "abs", "tabular", "Permutation allineato — tabulare, |attribution|"),
        ("canonical", "abs", "joint", "SHAP GradientExplainer — joint"),
        ("aligned_signed", "target", "tabular", "Permutation con segno — classe vera (`target`)")]:
    _t = _s[(_s.source == _src) & (_s["mode"] == _mode) & (_s.method == _meth)]
    if not len(_t):
        print(f"\n[{_title}] nessuna riga — sorgente non presente su disco.")
        continue
    print(f"\n{_title}   (deletion, q<={_QLOW}, gain_norm)")
    display(_t.pivot_table(index="impl_name", columns="stratum", values="gain_norm")
              .reindex(columns=[c for c in STRATA_SHOW if c in set(_t.stratum)]).round(3))

print("""
NOTA — `sure_wrong` e le celle sottili sono spesso vuote: sotto MIN_CELL_N il valore viene
trattenuto di proposito (la colonna `n` in `strata` dice quanti campioni ci sono davvero).
La confidenza e la classe NON sono lo stesso asse: il modello e' sicuro soprattutto sui
negativi, quindi le celle incrociate `pos_sure` / `neg_unsure` sono in `strata` e vanno
guardate prima di leggere un effetto di confidenza come tale.
""")

## ▶ Spiegazioni LORE — rigenerarle con il motore nuovo (`lore_sa`)

**Questa è l'unica parte che richiede GPU e tempo.** Tutto il resto del post-hoc gira sulle
curve già salvate.

Stato: `lore_sa` è il motore di default (`xai_driver.LORE_ENGINE`), ma **non ha mai prodotto un
artefatto**: le 111 directory `xai/lore/` sul disco vengono tutte dal motore *legacy*, il cui
vicinato `random` dà l'1.7% di evidenza di classe opposta — spiegazioni che non discriminano fra
strategie. Il motore nuovo scrive in sottocartelle **diverse** (`xai/lore_sa/` (fino al 31/07: `xai/lore/epoch*/tabular_sa`),
`lore_sa_per_epoch.csv`), quindi è **additivo**: non tocca il legacy e non serve `allow_clobber`.

> ⚠️ A differenza della sezione precedente, queste celle richiedono **tutto il setup** del
> notebook (dati, `FeatureSpace`, `xf.configure` alla cella "XAI engines"): il driver carica i
> checkpoint e perturba le feature vere.

**Costo**, misurato: ~0.58 min per campione, 32 campioni per run → **~19 min per run**.

| selezione | run | tempo |
| --- | ---: | ---: |
| `LORE_ONLY_IMPL = None`, `LORE_MAX_RUNS = 1` — prova | 1 | ~19 min |
| una run per famiglia (le 6 canoniche) | 6 | **~2 h** |
| solo `fixed_scaling` (il test acido su α) | 45 | ~14 h |
| tutte le run canoniche | 110 | **~34 h** |

In [ ]:
# @@LORE_SECTION@@ 1/2 — il run (GPU)
# ================================================================================
# Rigenera le spiegazioni LORE con `lore_sa`. Additivo: dal 2026-08-01 scrive in
# `<run_dir>/xai/lore_sa/` (prima era `xai/lore/epoch*/tabular_sa/`), quindi non tocca gli
# artefatti del motore legacy in `xai/lore/` — il write guard e' per-motore, verificato.
# La figura per-run la produce il callback in `xai/lore_sa/plots/`.
#
# Procedura: LORE_DRY_RUN=True per vedere cosa farebbe -> una run di prova -> la selezione
# vera. Riesecuzione = ripresa: le run gia' fatte le rifiuta il write guard e il loop prosegue.
#
# NB la selezione si costruisce QUI e non con `max_runs` del batch: `max_runs` fa head(n)
# sull'intero registro, e le prime 6 righe sono 5 seed di vanilla + 1 fixed_scaling, non
# una run per famiglia.
# ================================================================================
from isic24 import xai_driver as xd
from isic24 import xai_faithfulness as xf
import gc, keras

# GUARD — precondizioni, controllate PRIMA di caricare qualunque checkpoint.
# Senza questo il loop carica un modello per run e fallisce alla costruzione del callback,
# stampando lo stesso muro di testo N volte in pochi secondi: sembra un run finito presto.
try:
    _ = xf._CTX.validation_ds
except Exception:
    raise RuntimeError(
        "\n  Il contesto XAI non e' configurato in questo kernel.\n"
        "  Esegui le celle di SETUP 9 -> 23 in ordine (la 23 e' quella che chiama\n"
        "  `xf.configure(...)`, sezione «XAI engines»), poi torna qui.\n"
        "  Le celle 1-7 da sole NON bastano: servono training_df / validation_ds /\n"
        "  feature_space, che le celle 9-22 costruiscono."
    ) from None
print("[guard] contesto XAI configurato — ok")


assert xd.LORE_ENGINE == "lore_sa", f"LORE_ENGINE={xd.LORE_ENGINE!r}: il legacy non produce artefatti nuovi"

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if LORE_ONLY_IMPL:
    _m = _m[_m.impl_name.isin(LORE_ONLY_IMPL)]
if globals().get("LORE_ONLY_TAG"):
    _m = _m[_m.setting_tag.isin(LORE_ONLY_TAG)]
if LORE_PER_FAMILY:
    _keys = ["impl_name", "setting_tag"] if globals().get("LORE_ONLY_TAG") else ["impl_name"]
    _m = _m.groupby(_keys, sort=False).head(int(LORE_PER_FAMILY))
_dropped = []
if LORE_MAX_RUNS:
    _dropped = _m.iloc[int(LORE_MAX_RUNS):].run_id.tolist()   # serve ai due warning sotto
    _m = _m.head(int(LORE_MAX_RUNS))
_m = _m.reset_index(drop=True)

# 0.67 min/campione: MISURATO il 30/07 su vanilla seed_47 (21.4 min per 32
# campioni). Il 0.58 del bench del 29/07 veniva da n=2 campioni.
_min = len(_m) * xd.LORE_SA_KWARGS["max_explain_samples"] * 0.67
print(f"[LORE] motore={xd.LORE_ENGINE} {xd.LORE_SA_KWARGS['neigh_type']} "
      f"num_instances={xd.LORE_SA_KWARGS['num_instances']} mutpb={xd.LORE_SA_KWARGS['mutpb']}")
print(f"[LORE] {len(_m)} run x {xd.LORE_SA_KWARGS['max_explain_samples']} campioni "
      f"-> stima {_min/60:.1f} h ({_min:.0f} min)\n")
display(_m[["impl_name", "setting_tag", "seed", "run_id"]])

# LORE_MAX_RUNS taglia DOPO i filtri: se ne scarta qualcuna va detto due volte, qui e in
# coda. Il 31/07 la selezione 3c (3 run) e' diventata 1 run senza che si vedesse: il print
# di testa lo diceva, ma a cella finita si legge la CODA.
if _dropped:
    print(f"\n{'!'*72}\n  LORE_MAX_RUNS={LORE_MAX_RUNS} SCARTA {len(_dropped)} run che i"
          f" filtri avevano selezionato.\n  NON verranno eseguite. LORE_MAX_RUNS = None"
          f" per farle tutte.\n{'!'*72}\n")

# Trappola nota: la PRIMA riga di fixed_scaling e' `a0p000`, l'endpoint immagine-only dove il
# ramo tabellare e' scalato a zero. Li' `lore_coverage` vale 0 per costruzione: e' un controllo
# valido, NON un valore rappresentativo della famiglia.
_deg = _m[_m.setting_tag.astype(str).str.contains("a0p000")]
if len(_deg):
    print(f"\n⚠ {len(_deg)} run selezionate sono l'endpoint a0p000 (ramo tabellare azzerato): "
          f"utili come controllo, non come rappresentante di famiglia.")

if LORE_DRY_RUN:
    print("\n*** DRY RUN — non esegue niente. LORE_DRY_RUN=False per partire. ***")

lore_res = {}
for _i, _r in _m.iterrows():
    _rd = os.path.join(ART_PATH, str(_r.run_dir).split("artifacts/enhanced/")[-1])
    try:
        lore_res[_r.run_id] = xd.run_posthoc_xai(
            _rd, engines=("lore",), best_epoch=int(_r.best_epoch),
            best_val_auc=float(_r.best_val_auc), dry_run=LORE_DRY_RUN, verbose=True)
    except Exception as _e:
        lore_res[_r.run_id] = {"_run": f"failed: {type(_e).__name__}: {_e}"}
        print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}")
    finally:
        if not LORE_DRY_RUN:
            keras.backend.clear_session(); gc.collect()
    print(f"[{_i+1}/{len(_m)}] {_r.run_id}: {lore_res[_r.run_id]}", flush=True)

_ok = sum(1 for v in lore_res.values() if isinstance(v, dict) and "_run" not in v)
print(f"\n[LORE] {_ok}/{len(lore_res)} run senza errori")
if _dropped:
    print(f"\n⚠⚠ ATTENZIONE — eseguite {len(_m)} run, ma i filtri ne avevano selezionate "
          f"{len(_m)+len(_dropped)}: LORE_MAX_RUNS={LORE_MAX_RUNS} ne ha SCARTATE "
          f"{len(_dropped)}, MAI ESEGUITE:")
    for _d in _dropped:
        print(f"     - {_d}")
    print("     Rimedio: LORE_MAX_RUNS = None nella cella PARAMETERS, riesegui QUELLA e poi\n"
          "     questa. Le run gia' fatte le rifiuta il write guard ([FAIL] RuntimeError =\n"
          "     'gia' fatta', NON un errore vero) e il loop prosegue con le mancanti.")
if LORE_DRY_RUN:
    print("\n" + "="*72)
    print("  NESSUN FILE E' STATO SCRITTO — era una simulazione (LORE_DRY_RUN=True).")
    print("  Per eseguire davvero: metti LORE_DRY_RUN=False nella cella PARAMETERS,")
    print("  riesegui QUELLA cella, poi questa.")
    print("="*72)
else:
    _w = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore", "epoch*",
                                       "tabular_sa", "*.csv"), recursive=True))
    print(f"\n[LORE] {len(_w)} file per campione ora su disco.")


In [ ]:
# @@LORE_SECTION@@ 2/2 — leggere i risultati (CPU, nessuna GPU)
# ================================================================================
# Aggrega i CSV per campione che il run sopra ha scritto. Gira su quello che c'e':
# se il run e' parziale, mostra la parte fatta.
#
# `lore_coverage` = frazione di campioni con vicinato NON mono-classe. E' la misura di
# ATTENDANCE: `LoreEngineSA` congela l'embedding immagine e perturba solo il tabellare,
# quindi se nessuna perturbazione tabellare ribalta la predizione vuol dire che quella
# decisione la comanda il ramo immagine.
#
# NB si ricalcola qui da `neigh_n_classes` invece di leggere `lore_sa_coverage` dal CSV
# per epoca: quella colonna conta come "degenere" ANCHE i vicinati poco diversificati,
# due cose che qui vanno tenute separate.
# ================================================================================
# DUE percorsi: `xai/lore_sa/` e' dove scrive il motore dal 2026-08-01, `xai/lore/epoch*/
# tabular_sa/` e' dove ha scritto fino al 31/07. Gli artefatti storici non si spostano e non
# si riscrivono, quindi vanno letti anche loro.
#
# ⚠️ MA UNA RUN ALLA VOLTA. Cinque run hanno entrambi i file, e sono DUE CAMPIONI DIVERSI:
# lo storico e' n=32 con pos_rate 0.406, il nuovo n=48. Concatenarli fa una media fra due
# campioni incompatibili — `lore_coverage` di `vanilla` passava da 0.875 a 0.738 e
# `n_campioni` leggeva 80. E' [[run-is-the-unit-not-the-row]] rientrato dalla lista dei file.
# Quindi: il nuovo VINCE, lo storico entra solo per le run che il nuovo non ha.
# L'identita' della run la decide `isic24.run_ids`, MAI una ricostruzione a mano.
from isic24.run_ids import run_id_from_artifact

_nuovi = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore_sa", "epoch*",
                                       "lore_sa_persample_*.csv"), recursive=True))
_storici = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore", "epoch*",
                                         "tabular_sa", "lore_sa_persample_*.csv"),
                            recursive=True))
_visti = {run_id_from_artifact(f) for f in _nuovi}
_solo_storici = [f for f in _storici if run_id_from_artifact(f) not in _visti]
_lore_files = _nuovi + _solo_storici
print(f"[LORE] {len(_lore_files)} run: {len(_nuovi)} da xai/lore_sa/ (nuovo), "
      f"{len(_solo_storici)} da xai/lore/.../tabular_sa/ (storico, n=32)"
      + (f" — {len(_storici) - len(_solo_storici)} file storici scartati perche' la run "
         f"ha gia' il nuovo" if len(_storici) != len(_solo_storici) else ""))
if _solo_storici and _nuovi:
    print("       ⚠ le due sorgenti hanno n e prevalenza DIVERSI: confrontare "
          "l'ordinamento, non i livelli, finche' non sono tutte rifatte.")

if not _lore_files:
    print("Nessun artefatto `lore_sa` ancora: esegui la cella sopra con LORE_DRY_RUN=False.")
else:
    # L'identita' della run la decide `isic24.run_ids`, MAI una ricostruzione a mano:
    # `run_id` fonde <study_tag>/seed_<seed> con <setting_tag>__e<epochs> usando `__`, quindi
    # contiene sia '/' sia due '__' e non e' ri-parsabile a occhio. Rifarlo qui e' il bug che
    # il 30/07 mandava tutte le run nel gruppo "?" collassando le 6 famiglie in una riga.
    _master = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
    _meta = {str(r.run_id): (r.impl_name, r.setting_tag) for _, r in _master.iterrows()}

    _rows, _unmatched = [], []
    for f in _lore_files:
        _rid = run_id_from_artifact(f)
        if _rid not in _meta:
            _unmatched.append(f"{_rid}   ({f})")
            continue
        d = pd.read_csv(f)
        d["run_id"] = _rid
        d["impl_name"], d["setting_tag"] = _meta[_rid]
        _rows.append(d)
    if _unmatched:
        raise RuntimeError("artefatti LORE non presenti nel registro canonico:\n  "
                           + "\n  ".join(_unmatched))

    L = pd.concat(_rows, ignore_index=True)
    L["covered"] = L["neigh_n_classes"] >= 2

    fam = L.groupby(["impl_name", "setting_tag"]).agg(
        n_run=("run_id", "nunique"),
        n_campioni=("i", "size"),
        lore_coverage=("covered", "mean"),                       # <- segnale di attendance
        neigh_pos_rate=("neigh_pos_rate", "mean"),
        neigh_unique_frac=("neigh_unique_frac", "mean"),
        rule_len=("rule_len", "mean"),
        fid_cv_bacc=("fid_cv_bacc", "mean"),
    )
    # La fedelta' ha senso solo dove il vicinato ha due classi: altrove e' indefinita.
    fam["n_covered"] = L.groupby(["impl_name", "setting_tag"])["covered"].sum()
    fam["fid_cv_bacc_covered"] = (L[L.covered]
                              .groupby(["impl_name", "setting_tag"])["fid_cv_bacc"].mean())
    print("\nqualita' delle spiegazioni LORE per famiglia "
          "(`lore_coverage` = quanto il ramo TABELLARE comanda la decisione)")
    display(fam.round(3))


    # Punto 6 della lista — class-wise sulle SPIEGAZIONI, con il RUN come unita'.
    #
    # ATTENZIONE, e' costato caro il 31/07: le run condividono TUTTE lo stesso campione di
    # validation. Quindi 10 run x 32 campioni non sono 320 osservazioni ma 32 lesioni viste
    # da 10 modelli, e un chi2 sulle righe poolate da' un p-value ottimistico -- dava
    # p=0.0016 dove il test corretto da' Wilcoxon p=0.047. Un valore per run, poi segni e
    # Wilcoxon FRA run.
    cw = (L.groupby(["impl_name", "setting_tag", "true_label"])
            .agg(n=("i", "size"), lore_coverage=("covered", "mean"),
                 neigh_pos_rate=("neigh_pos_rate", "mean"),
                 rule_len=("rule_len", "mean"), fid_cv_bacc=("fid_cv_bacc", "mean"))
            .unstack("true_label"))
    print("\nle stesse quantita' separate per classe vera (0 = benigno, 1 = maligno)")
    display(cw.round(3))
    cw.reset_index().to_csv(
        os.path.join(RUNS_PATH, "_faithfulness", "lore_sa_quality_by_family_class.csv"),
        index=False)

    print("\nil contrasto fra classi, testato correttamente:")
    _per = L.groupby(["run_id", "true_label"])["covered"].mean().unstack().dropna()
    if len(_per) >= 2:
        xs.print_across_runs(xs.across_runs(_per[0], _per[1], "benigni", "maligni"))
    else:
        print("  servono almeno 2 run con entrambe le classi.")

    _out = os.path.join(RUNS_PATH, "_faithfulness", "lore_sa_quality_by_family.csv")
    fam.reset_index().to_csv(_out, index=False)
    print("scritto", _out)

    print("""
COME LEGGERLA:
  lore_coverage alta  -> le feature tabellari da sole ribaltano la predizione: il ramo
                         tabellare pesa. Bassa -> comanda l'immagine, e LORE su quei
                         campioni non ha niente da spiegare.
  fid_cv_bacc         -> leggerla SEMPRE accanto a neigh_unique_frac: dove la diversita'
                         e' bassa e' ottimisticamente distorta. E mai `fid_insample`, che
                         vale 1.0 per costruzione.

TEST ACIDO (serve la famiglia fixed_scaling, alpha da 0.0 a 1.0):
  alpha=0.0 -> ramo tabellare azzerato -> lore_coverage deve ess
  ere ~0
  alpha=1.0 -> solo tabellare          -> lore_coverage deve essere alta
  Se non separa questi due estremi, la metrica non misura l'attendance e va buttata.
  Se li separa, la domanda vera e' se ordina anche il mezzo — ed e' li' che il legacy
  aveva fallito.
""")

In [ ]:
print(LORE_DRY_RUN, LORE_ONLY_IMPL, LORE_PER_FAMILY, LORE_MAX_RUNS)

## ▶ Campione stratificato — ⚠️ la motivazione originale era SBAGLIATA

> **Ritirato il 31/07, lo stesso giorno.** Questa sezione era nata per correggere una presunta
> distorsione del prefisso: «i primi 32 campioni hanno il 40.6% di maligni contro il 9.1% del
> validation». **Quel 9.1% è sbagliato**: veniva da `neg_sample = 0.01` senza la riga successiva,
> **`pos_sample = 5.0`** — i positivi sono upsamplati 5×. La prevalenza vera è **0.334**, e il
> prefisso a 0.406 su n=32 dà **P = 0.245**: rumore. In più `df.sample(frac=1.0)` rimescola prima
> dello split in fold, quindi «le prime N righe» sono già un campione casuale.
>
> **Non c'è nessuna distorsione del prefisso da correggere.**

### Il problema vero, che invece resta

`pos_sample=5.0` usa `replace=True`: **364 positivi unici diventano 1820 righe**. Sul campione
n=256 costruito qui ci sono **222 lesioni uniche su 256**, e fra gli 85 positivi solo **51
lesioni uniche** (molteplicità fino a 3). La XAI spiega la stessa lesione più volte: i campioni
non sono indipendenti e **ogni CI per run è ottimistica** — l'n efficace sui positivi è circa
metà di quello nominale.

Il rimedio è **deduplicare su `isic_id`**, non stratificare. Non è ancora implementato.

### Cosa vale ancora la pena eseguire qui

- La **`1/4`** e la **`2/4`**: costruiscono un campione con identità verificabile e misurano di
  quanto si sposta il gain cambiando campione. Resta la risposta utile alla domanda «i numeri
  dipendono da *quali* 128 righe?», che è legittima anche senza bias.
- Il **guard sull'impronta** in `_load_shap_values`: gli npz salvano `explained_indices` come
  *posizioni*, quindi riusarli sotto un campione diverso accoppia i valori SHAP di certe righe
  alle feature di altre lesioni — senza errore e con numeri plausibili. Rischio reale
  qualunque campione si scelga.
- Le **`3/4`** e **`4/4`** non sono più urgenti: non c'è un bias da correggere sulle 110 run
  canoniche. Restano il modo per rifare i conti su un campione diverso quando servirà.

Tutto scrive in `runs/_stratified/` e non tocca un solo artefatto canonico.

In [ ]:
# @@SUBSET@@ 0/3 — TEST: lo stampigliatore SHAP fa round-trip? (GPU, ~3 min, UNA run)
# ================================================================================
# L'unico pezzo mai verificato end-to-end: `ShapCallback` scrive `subset_fingerprint`
# nell'npz e `_load_shap_values` lo riaccetta? Il guard era stato testato con un npz
# stampigliato a mano. ESEGUIRE QUESTO PRIMA di qualunque giro lungo.
from isic24 import xai_driver as xd
import numpy as np, gc, keras, tempfile, shutil

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_r = _m[_m.impl_name == "vanilla"].iloc[0]
_rd = os.path.join(ART_PATH, str(_r.run_dir).split("artifacts/enhanced/")[-1])
_tmp = tempfile.mkdtemp(prefix="shap_roundtrip_")
print(f"run di prova: {_r.run_id}\nscrivo in {_tmp} (temporanea, cancellata alla fine)\n")

_ok = {}
try:
    # 1) SHAP sul campione adottato, in una dir temporanea (mai dentro le run canoniche)
    xd.run_posthoc_xai(_rd, engines=("shap",), best_epoch=int(_r.best_epoch),
                       best_val_auc=float(_r.best_val_auc),
                       xai_dir=os.path.join(_tmp, "xai"), verbose=False)
    _npz = sorted(glob.glob(os.path.join(_tmp, "xai", "shap", "epoch_*", "*.npz")))
    _z = np.load(_npz[0])
    _stamp = str(_z["subset_fingerprint"]) if "subset_fingerprint" in _z.files else None
    _ok["1. npz stampigliato"] = (_stamp == XAI_SUBSET_FP, f"{_stamp}")
    _ok["2. n righe = campione"] = (_z["shap_values"].shape[0] == len(xai_subset_idx),
                                    f"{_z['shap_values'].shape[0]} vs {len(xai_subset_idx)}")
    # 3) il guard lo RIACCETTA
    _sv, _ei = xf._load_shap_values(_tmp, int(_r.best_epoch), None,
                                    pd.DataFrame({"a": range(len(xai_subset_idx))}), None)
    _ok["3. guard lo riaccetta"] = (_sv.shape[0] == len(xai_subset_idx), f"{_sv.shape}")
    # 4) e RIFIUTA un npz canonico (senza stampigliatura) sotto lo stesso campione
    try:
        xf._load_shap_values(_rd, int(_r.best_epoch), None,
                             pd.DataFrame({"a": range(300)}), None)
        _ok["4. rifiuta il canonico"] = (False, "NON ha sollevato")
    except RuntimeError as _e:
        _ok["4. rifiuta il canonico"] = ("SUBSET GUARD" in str(_e), "RuntimeError")
finally:
    keras.backend.clear_session(); gc.collect()
    shutil.rmtree(_tmp, ignore_errors=True)

print()
for _k, (_p, _d) in _ok.items():
    print(f"  {'PASS' if _p else '** FAIL **'}  {_k}   ({_d})")
print("\n" + ("TUTTO PASS — si puo' procedere." if all(p for p, _ in _ok.values())
              else "*** FERMARSI QUI: SHAP non e' riutilizzabile su questo campione. ***"))

In [ ]:
# @@SUBSET@@ 1/3 — la SONDA: quanto si sposta il numero? (GPU, ~8 min, UNA run)
# ================================================================================
# Stessa run, stesso attributore, DUE campioni: il prefisso storico e le 144 lesioni
# distinte. Se il gain si sposta poco, la correzione e' dovuta ma di conseguenza numerica
# piccola; se si sposta molto, meglio saperlo adesso che dopo due ore.
#
# ⚠️ DUE TRAPPOLE, entrambe gia' scattate. Leggerle prima di modificare questa cella.
#
# 1. `subset_idx=None` NON vuol dire "il prefisso". `xs.adopt()` nel setup imposta il
#    globale `xf.FAITH_SUBSET_IDX`, e `_build_faith_subset` ci ricade ogni volta che
#    subset_idx e' None (e' il design: "da li' in poi nessuna cella deve passare
#    subset_idx"). Il 31/07 la sonda ha girato COSI': i due bracci hanno spiegato le
#    STESSE 144 righe, labels identiche e |Delta pred| = 0.0 esatto, e il verdetto
#    "non sposta" era il confronto di un campione con se stesso. Il braccio di
#    riferimento passa ora `np.arange(n)` ESPLICITO, e il guard in coda rifiuta di
#    stampare un verdetto se i due bracci coincidono ancora.
#
# 2. Le MODES. Con modes=("abs",) ogni gain vale ~+0.01 (l'artefatto del ranking
#    assoluto, cfr. FINDINGS_signed_tabular) e una soglia di "rilevante" a 0.10 non e'
#    raggiungibile per costruzione: il test non puo' fallire. Qui NON si passa `modes`,
#    quindi girano le stesse quattro della cella 2/3. I modi con segnale stanno a
#    +0.85…+0.90 ed e' li' che uno spostamento si vede.
#
# COSA VARIA FRA I DUE CAMPIONI: due cose insieme. (a) via i duplicati (nel prefisso la
# stessa lesione ricorre fino a ~5 volte); (b) cambia la prevalenza (il campione nuovo e'
# 50/50 per costruzione, il prefisso e' quello che e' — la cella la stampa per entrambi).
# La classe e' un asse reale della fedelta' (cfr. faithfulness_strata), quindi il delta
# NON e' "l'effetto dei duplicati": e' l'effetto di passare dal prefisso a 144 lesioni
# distinte bilanciate.
from isic24 import xai_driver as xd
import gc, keras, time, tempfile, shutil

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_r = _m[(_m.impl_name == "vanilla")].iloc[0]
_rd = os.path.join(ART_PATH, str(_r.run_dir).split("artifacts/enhanced/")[-1])
# La sonda e' diagnostica: i suoi `probe_*` non sono artefatti da tenere e dentro
# `xai/shap_tab/` sarebbero solo rumore. Dir temporanea, cancellata in fondo.
_out = tempfile.mkdtemp(prefix="probe_subset_")
_REF, _NEW = "PREFISSO", "LESIONI DISTINTE"
_N = len(xai_subset_idx)
_ARMS = [(_REF, "prefisso", np.arange(_N)),      # <- esplicito: NON ereditare il default
         (_NEW, "distinte", np.asarray(xai_subset_idx))]
print(f"sonda su {_r.run_id}  |  modes = {xf.SIGNED_MODES}  |  n = {_N}")
print(f"  indici in comune fra i due bracci: "
      f"{len(np.intersect1d(_ARMS[0][2], _ARMS[1][2]))}/{_N}\n")

_model, _ = xd.load_run_model(_rd)
_res = {}
for _tag, _pfx, _si in _ARMS:
    _t0 = time.time()
    # NB ritorna (results, phi), non un dict: spacchettare, non fare .items() sulla tupla.
    # `phi` si calcola UNA volta e vale per tutte le modes: aggiungerle costa un passo di
    # perturbazione ciascuna, non un secondo passo di attribuzione.
    _o, _ = xf.aligned_signed_faithfulness(
        _model, _rd, int(_r.best_epoch), subset_idx=_si,
        out_dir=_out, out_prefix=f"probe_{_pfx}", verbose=False,
        q_maxes=tuple(Q_MAX_REPORT))
    _res[_tag] = {k: v for k, v in _o.items() if "gain" in str(k) and isinstance(v, float)}
    print(f"  {_tag:17s} {time.time()-_t0:5.0f}s", flush=True)
keras.backend.clear_session(); gc.collect()

# GUARD — i due bracci hanno visto campioni DIVERSI? Le curve per campione portano labels e
# predizioni: se coincidono, la sonda ha confrontato un campione con se stesso e ogni delta
# sotto vale zero per costruzione. Fermarsi qui, non leggere la tabella.
_z = {t: np.load(os.path.join(_out, f"probe_{p}_target_curves.npz")) for t, p, _ in _ARMS}
_same = np.array_equal(_z[_REF]["preds_orig"], _z[_NEW]["preds_orig"])
print(f"\n  prevalenza positivi: {_REF} {_z[_REF]['labels'].mean():.3f}  |  "
      f"{_NEW} {_z[_NEW]['labels'].mean():.3f}")
if _same:
    raise RuntimeError(
        "\n  I DUE BRACCI HANNO GIRATO SULLO STESSO CAMPIONE (preds_orig identiche).\n"
        "  Causa nota: `subset_idx=None` eredita `xf.FAITH_SUBSET_IDX` impostato da\n"
        "  `xs.adopt()`. Il braccio di riferimento deve passare np.arange(n) esplicito.\n"
        "  Nessun delta sotto sarebbe interpretabile.")
print("  [guard] i due bracci sono campioni diversi — ok")

_cmp = pd.DataFrame({_REF: _res[_REF], _NEW: _res[_NEW]}).dropna()
_cmp.index = [str(i).split("aligned_")[-1] for i in _cmp.index]
_cmp["delta"] = _cmp[_NEW] - _cmp[_REF]
_cmp["rel_%"] = 100 * _cmp["delta"] / _cmp[_REF].abs().where(_cmp[_REF].abs() > 1e-9)
# `abs` vive a ~0.01 e oscilla di ~0.010 fra esecuzioni identiche: il suo rel_% e' rumore
# diviso rumore. Marcarlo, cosi' nessuno lo legge come uno spostamento.
_cmp["rilevante"] = np.where(_cmp.index.str.startswith("abs"), "(abs: sotto il rumore)",
                     np.where(_cmp["delta"].abs() > 0.10, "<- SPOSTA", ""))
print(f"\n  spostamento ({_NEW.lower()} - {_REF.lower()}):")
display(_cmp.round(3))
_big = _cmp[(_cmp["delta"].abs() > 0.10) & (~_cmp.index.str.startswith("abs"))]
print(f"\n  verdetto: {'SPOSTA' if len(_big) else 'non sposta'} i modi con segnale "
      f"({len(_big)} metriche non-abs oltre 0.10 in assoluto)")
shutil.rmtree(_out, ignore_errors=True)


In [24]:
# @@SUBSET@@ 1/4 — RIFACIMENTO COMPLETO della XAI sul campione a lesioni distinte
# ================================================================================
from isic24 import xai_driver as xd
import gc, keras, time, os, numpy as np, pandas as pd

JOBS_TO_RUN = ["lore"] # ["gradcam", "shap", "lore"]
ONLY_IMPL   = None        # es. ["vanilla"]
ONLY_SEEDS  = None        # es.
PER_FAMILY  = 1        # es. 1
LORE_N      = 48          # LORE costa 29 s/campione: 48 su 110 run sono ~42 h
DRY_RUN     = False
LORE_ON_CPU = True

xd.LORE_SA_DEVICE = "/CPU:0" if LORE_ON_CPU else None

# GUARD — senza campione adottato i motori scriverebbero nelle cartelle CANONICHE.
assert xd.shap_subdir() == "shap_tab" and xd.gradcam_subdir() == "gradcam_img", (
    "nessun campione adottato in questo kernel: esegui la cella di SETUP con `xs.adopt(...)`.\n"
    "  Senza, SHAP e Grad-CAM scriverebbero sopra gli artefatti canonici del prefisso.")

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
# I seed sono NUMERI nel master: `["47"]` non seleziona niente e la cella sembrerebbe
# non avere lavoro da fare. Si convertono invece di far fallire il confronto.
if ONLY_SEEDS is not None: ONLY_SEEDS = [int(_s) for _s in ONLY_SEEDS]
if ONLY_IMPL:  _m = _m[_m.impl_name.isin([str(_x) for _x in ONLY_IMPL])]
if ONLY_SEEDS: _m = _m[_m.seed.astype(int).isin(ONLY_SEEDS)]
if PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(PER_FAMILY))
# GUARD — una selezione VUOTA non e' un giro riuscito. Senza questo la cella stampava
# `ok 0/0` e "chiusura ✓" senza aver fatto niente: il modo piu' facile di credere che una
# notte di calcolo sia andata bene quando non e' nemmeno partita.
_all = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if not len(_m):
    raise ValueError(
        f"\n  I filtri non selezionano nessuna run.\n"
        f"    ONLY_IMPL  = {ONLY_IMPL}      valori possibili: {sorted(_all.impl_name.unique())}\n"
        f"    ONLY_SEEDS = {ONLY_SEEDS}      valori possibili: {sorted(_all.seed.unique())}\n"
        f"    PER_FAMILY = {PER_FAMILY}\n"
        f"  (ONLY_IMPL vuole un nome di FAMIGLIA, non di motore: i motori si scelgono con "
        f"JOBS_TO_RUN.)")
_m = _m.reset_index(drop=True)
_RD = {r.run_id: os.path.join(ART_PATH, str(r.run_dir).split("artifacts/enhanced/")[-1])
       for _, r in _m.iterrows()}
print(f"selezionate {len(_m)} run: {sorted(_m.impl_name.unique())} | seed {sorted(_m.seed.unique())}")

# Il campione di LORE: meta' per classe, a passo costante DENTRO ogni classe, cosi' non e'
# a sua volta un prefisso (troncare i 144 ordinati da' 35 pos / 13 neg — misurato).
_y = validation_df.target.values[np.asarray(xai_subset_idx)]
_lore_idx = np.sort(np.concatenate([
    np.asarray(xai_subset_idx)[_y == c][
        np.linspace(0, int((_y == c).sum()) - 1, LORE_N // 2).round().astype(int)]
    for c in (0, 1)]))
assert len(_lore_idx) == LORE_N and validation_df.target.values[_lore_idx].sum() == LORE_N // 2

# ---- il piano, PRIMA di occupare la GPU ----------------------------------------------
print(f"campione XAI: {XAI_SUBSET_FP}")
print(f"  SHAP e Grad-CAM su {len(xai_subset_idx)} lesioni | LORE su {len(_lore_idx)} "
      f"({LORE_N//2} pos / {LORE_N//2} neg)\n")
_plan, _tot = {}, 0.0
for _j in JOBS_TO_RUN:
    _cfg = xs.JOBS[_j]
    _n = len(_lore_idx) if _j == "lore" else len(xai_subset_idx)
    _exp = _lore_idx if _j == "lore" else np.asarray(xai_subset_idx)
    _todo, _done, _redo = xs.resume_verified(_RD, _cfg["kind"], XAI_SUBSET_FP, _exp,
                                             task=_cfg["task"])
    _work = _todo + _redo               # `_redo` = stampigliate ma su un altro campione
    _sec = _cfg["sec_per_sample"]
    _h = (len(_work) * _n * _sec / 3600) if _sec else None
    _tot += _h or 0
    _plan[_j] = (_work, set(_redo), _cfg, _n)
    print(f"  {_j:8s} -> xai/{_cfg['kind']:12s} ok {len(_done):3d} | da fare {len(_todo):3d}"
          + (f" | DA RIFARE {len(_redo):3d} (altro campione)" if _redo else "")
          + (f" | ~{_h:.1f} h ({_n*_sec/60:.1f} min/run)" if _h else " | costo DA MISURARE"))
print(f"\n  TOTALE STIMATO: ~{_tot:.1f} h" + ("  (+ Grad-CAM, non misurato)"
      if any(xs.JOBS[j]["sec_per_sample"] is None for j in JOBS_TO_RUN) else ""))
print("  Interrompere quando si vuole: al rilancio riparte da dove si e' fermato.")

if DRY_RUN:
    print("\n*** DRY RUN — niente eseguito. DRY_RUN=False per partire. ***")
else:
    _t0, _orfani = time.time(), []
    for _j in JOBS_TO_RUN:
        _todo, _redo_set, _cfg, _n = _plan[_j]
        if not _todo:
            continue
        print(f"\n===== {_j.upper()} — {len(_todo)} run =====", flush=True)
        if _j == "lore":
            xd.LORE_SA_KWARGS["max_explain_samples"] = len(_lore_idx)
            xd.LORE_SA_KWARGS["subset_idx"] = _lore_idx
            print(f"  (LORE su {xd.LORE_SA_DEVICE or 'device di default = GPU'})",
                  flush=True)
        for _k, _rid in enumerate(_todo):
            _rd, _row = _RD[_rid], _m[_m.run_id == _rid].iloc[0]
            _t1 = time.time()
            try:
                # `allow_clobber` SOLO per le run da rifare: quegli artefatti li ha
                # scritti questa stessa pipeline su un campione diverso, e sostituirli e'
                # il punto. Sulle altre il guard resta armato, cosi' un artefatto orfano
                # (scritto da un tentativo ucciso prima della stampigliatura) viene
                # segnalato invece che sovrascritto di nascosto.
                _o = xd.run_posthoc_xai(_rd, engines=_cfg["engines"],
                                        best_epoch=int(_row.best_epoch),
                                        best_val_auc=float(_row.best_val_auc),
                                        overwrite=True, verbose=False,
                                        allow_clobber=(_rid in _redo_set))
                # ⚠️ `run_posthoc_xai` NON solleva se un motore fallisce: `run_xai_on_best
                # _epoch` cattura l'eccezione e la mette nel dict di ritorno come
                # "failed: ...". Senza questo controllo la stampigliatura veniva scritta
                # anche su una run fallita, e la ripresa l'avrebbe saltata per sempre —
                # successo davvero il 01/08 sull'OOM del Grad-CAM. Su un giro di 42 ore
                # significherebbe scoprire i buchi solo alla fine.
                _bad = {k: v for k, v in (_o or {}).items() if str(v) != "ok"}
                if _bad or not _o:
                    print(f"  [FAIL] {_rid}: {_bad or 'nessun motore ha girato'}", flush=True)
                    continue
                xs.mark_done(_rd, _cfg["kind"], XAI_SUBSET_FP, task=_cfg["task"],
                             payload={"run_id": _rid, "impl_name": _row.impl_name,
                                      "setting_tag": _row.setting_tag, "n": int(_n),
                                      "result": str(_o)})
                print(f"  [{_k+1}/{len(_todo)}]{' RIFATTA' if _rid in _redo_set else ''} "
                      f"{_rid}: {_o}  "
                      f"({(time.time()-_t1)/60:.1f} min | {(time.time()-_t0)/60:.0f} tot)",
                      flush=True)
            except RuntimeError as _e:
                if "WRITE GUARD" in str(_e):
                    # L'unico caso in cui il guard litiga con la ripresa: artefatti scritti
                    # da un tentativo precedente ucciso PRIMA della stampigliatura. Non si
                    # sovrascrive di nascosto — si segnala e si va avanti.
                    _orfani.append((_j, _rid, os.path.join(_rd, "xai", _cfg["kind"])))
                    print(f"  [ORFANI] {_rid}: artefatti {_j} senza stampigliatura", flush=True)
                else:
                    print(f"  [FAIL] {_rid}: {type(_e).__name__}: {_e}", flush=True)
            except Exception as _e:
                print(f"  [FAIL] {_rid}: {type(_e).__name__}: {_e}", flush=True)
            finally:
                keras.backend.clear_session(); gc.collect()
        if _j == "lore":
            xd.LORE_SA_KWARGS.pop("subset_idx", None)

    print(f"\n===== chiusura ({(time.time()-_t0)/60:.0f} min) =====")
    for _j in JOBS_TO_RUN:
        _cfg = xs.JOBS[_j]
        _exp = _lore_idx if _j == "lore" else np.asarray(xai_subset_idx)
        _todo, _done, _redo = xs.resume_verified(_RD, _cfg["kind"], XAI_SUBSET_FP, _exp,
                                                 task=_cfg["task"])
        print(f"  {_j:8s} ok {len(_done)}/{len(_m)} sul campione giusto"
              + (f" — restano {len(_todo)+len(_redo)}, rilancia questa cella"
                 if (_todo or _redo) else "  ✓"))
    if _orfani:
        print(f"\n⚠ {len(_orfani)} run con artefatti ORFANI (scritti da un tentativo ucciso "
              f"prima della stampigliatura). Il write guard rifiuta di sovrascriverli.\n"
              f"  Per rifarli, cancellare la cartella e rilanciare:")
        for _j, _rid, _p in _orfani[:10]:
            print(f"    rm -rf {_p}    # {_j} {_rid}")


selezionate 6 run: ['adaptive_gating', 'controlled_fusion', 'fixed_scaling', 'learnable_scaling', 'loss_regularization', 'vanilla'] | seed [np.int64(8), np.int64(47)]
campione XAI: unique-equal-s0-n144-5671a4e59ff6998a
  SHAP e Grad-CAM su 144 lesioni | LORE su 48 (24 pos / 24 neg)

  lore     -> xai/lore_sa      ok   0 | da fare   4 | DA RIFARE   2 (altro campione) | ~2.3 h (23.2 min/run)

  TOTALE STIMATO: ~2.3 h
  Interrompere quando si vuole: al rilancio riparte da dove si e' fermato.

===== LORE — 6 run =====
  (LORE su /CPU:0)


I0000 00:00:1785630731.345234  934407 service.cc:148] XLA service 0x7417d4012270 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785630731.345282  934407 service.cc:156]   StreamExecutor device (0): Host, Default Version
2026-08-02 02:32:11.473928: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1785630732.838255  934407 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  [1/6] 20260622-112038/seed_47__a0p000__e20: {'LoreSACallback': 'ok'}  (19.7 min | 20 tot)
  [2/6] 20260619-184157/seed_47__init_a0p100__e20: {'LoreSACallback': 'ok'}  (19.8 min | 40 tot)


2026-08-02 03:11:50,561 tensorflow   WARNING  5 out of the last 5 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7418d84ea840> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


2026-08-02 03:11:54,974 tensorflow   WARNING  6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7418d84ea840> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


  [3/6] 20260610-095420/seed_47__lam0e+00__e20: {'LoreSACallback': 'ok'}  (20.0 min | 60 tot)
  [4/6] 20260616-102253/seed_8__base__e20: {'LoreSACallback': 'ok'}  (20.0 min | 80 tot)
  [5/6] RIFATTA 20260604-210304/seed_47__baseline__e20: {'LoreSACallback': 'ok'}  (20.1 min | 100 tot)
  [6/6] RIFATTA 20260629-185216/seed_47__base__e20: {'LoreSACallback': 'ok'}  (20.6 min | 120 tot)

===== chiusura (120 min) =====
  lore     ok 6/6 sul campione giusto  ✓


In [ ]:
# @@SUBSET@@ 2/3 — fedelta' tabellare allineata sul campione a lesioni distinte (GPU)
# ================================================================================
# COSTO — misurato il 31/07 su 5 run vanilla: 18 min, cioe' ~3.6 min/run a n=144, cioe'
# 1.5 s/campione. Il vecchio 0.52 s/campione era la cifra dell'attributore a UN modo: qui
# `aligned_signed_faithfulness` gira le QUATTRO modes di SIGNED_MODES, e ogni modo e' un
# passo di perturbazione in piu' (l'attribuzione, quella si', si calcola una volta sola).
#   Non rigenera SHAP: l'attributore allineato calcola tutto al volo.
#   Con i knob committati (tutti None) sono TUTTE le 110 run = ~6.6 h.
#   La ripresa e' per-run: si puo' interrompere e rilanciare, le fatte si saltano.
#
# DOVE SCRIVE (cambiato il 2026-08-01): dentro la run, in `<run_dir>/xai/shap_tab/`.
# Non esiste piu' la radice parallela `runs/_subset/`. La separazione dai canonici la fa il
# NOME della cartella: `xai/faithfulness/` (numeri pubblicati) resta intatta e il SUBSET
# GUARD in `xai_faithfulness` rifiuta di scriverci dentro finche' un campione e' adottato.
# Le FIGURE per-run le produce il motore in `xai/shap_tab/plots/`, e il rendiconto fra
# famiglie lo fa `xs.report(...)` in coda a questa cella: nessuna cella in piu' da lanciare.
from isic24 import xai_driver as xd
import gc, keras, time, os, pandas as pd

STRAT_ONLY_IMPL  = None         # None = tutte le famiglie
STRAT_ONLY_SEEDS = None            # es. [47] per una run per famiglia; None = tutti i seed
STRAT_PER_FAMILY = None            # es. 1 per una sola run per famiglia
STRAT_DRY_RUN    = False            # <- False per partire

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if STRAT_ONLY_IMPL:  _m = _m[_m.impl_name.isin(STRAT_ONLY_IMPL)]
if STRAT_ONLY_SEEDS: _m = _m[_m.seed.isin(STRAT_ONLY_SEEDS)]
if STRAT_PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(STRAT_PER_FAMILY))
_m = _m.reset_index(drop=True)

# run_id -> run_dir, riancorato su ART_PATH: il master porta i path della macchina su cui e'
# stato scritto. La ripresa lavora su questa mappa, non piu' su una radice separata.
_RD = {r.run_id: os.path.join(ART_PATH, str(r.run_dir).split("artifacts/enhanced/")[-1])
       for _, r in _m.iterrows()}

# 1.5 s/campione = misurato, con le 4 modes. Cambiare SIGNED_MODES cambia questa costante.
SEC_PER_SAMPLE = 1.5
_todo, _done = xs.resume_report(_RD, "shap_tab", XAI_SUBSET_FP)
_est = len(_todo) * len(xai_subset_idx) * SEC_PER_SAMPLE
print(f"[subset] {len(_m)} run selezionate | gia' fatte {len(_done)} | da fare {len(_todo)}")
print(f"[subset] {len(xf.SIGNED_MODES)} modes x {len(xai_subset_idx)} campioni "
      f"-> stima {_est/60:.0f} min ({_est/3600:.1f} h)  [{_est/60/max(len(_todo),1):.1f} min/run]")
if _done:
    print(f"        (ripresa attiva: le {len(_done)} gia' fatte con l'impronta {XAI_SUBSET_FP} sono saltate)")
if STRAT_DRY_RUN:
    print("\n*** DRY RUN — niente eseguito. STRAT_DRY_RUN=False per partire. ***")
else:
    _t0 = time.time()
    for _i, _r in _m.iterrows():
        _rd = _RD[_r.run_id]
        if xs.is_done(_rd, "shap_tab", XAI_SUBSET_FP):
            continue
        try:
            _model, _ = xd.load_run_model(_rd)
            # (results, phi): spacchettare. Con `_o = ...` il dict comprehension sotto
            # sollevava AttributeError DOPO il calcolo, l'except lo inghiottiva come [FAIL]
            # e `mark_done` non veniva mai raggiunta: ogni run costava i suoi minuti e non
            # lasciava stampigliatura, quindi la ripresa rifaceva tutto da capo.
            # out_dir non si passa piu': lo risolve la funzione in xai/shap_tab/.
            _o, _ = xf.aligned_signed_faithfulness(
                _model, _rd, int(_r.best_epoch), subset_idx=xai_subset_idx,
                verbose=False, q_maxes=tuple(Q_MAX_REPORT))
            # La stampigliatura si scrive SOLO qui, a calcolo finito: e' cio' che rende
            # sicura l'interruzione (una run a meta' non lascia stampigliatura e si rifa').
            xs.mark_done(_rd, "shap_tab", XAI_SUBSET_FP,
                         {"run_id": _r.run_id, "impl_name": _r.impl_name,
                          "setting_tag": _r.setting_tag, "n": len(xai_subset_idx),
                          "gains": {k: v for k, v in _o.items() if isinstance(v, float)}})
            print(f"[{_i+1}/{len(_m)}] {_r.run_id}: ok  ({(time.time()-_t0)/60:.0f} min totali)", flush=True)
        except Exception as _e:
            print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}", flush=True)
        finally:
            keras.backend.clear_session(); gc.collect()
    _todo, _done = xs.resume_report(_RD, "shap_tab", XAI_SUBSET_FP)
    print(f"\n[subset] fatte {len(_done)}/{len(_m)} in {(time.time()-_t0)/60:.0f} min"
          + (f" — restano {len(_todo)}, rilancia questa cella per riprendere" if _todo else ""))

    # Rendiconto fra famiglie, in coda e senza GPU. `central` NON e' opzionale: il master ha
    # 45 righe di fixed_scaling, `a0p000` (ramo tabellare azzerato, tutti zeri) compresa, e
    # una media di famiglia che le mescola tutte non vuol dire niente.
    if not _todo:
        # `tag` non si passa: `report` lo deriva dall'`n` dei dati (`_n144` qui, `_n48`
        # per LORE), cosi' i due kind non possono etichettarsi con l'n dell'altro.
        # `encoded_feature_names` accende anche le metriche di forma CROSS-RUN (stabilita'
        # fra configurazioni, accordo SHAP<->LORE): quelle per-run le hanno gia' scritte i
        # motori mentre calcolavano, queste due hanno bisogno di tutte le run insieme.
        xs.report(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                  ART_PATH, kinds=("shap_tab",), central=CENTRAL_SETTINGS,
                  encoded_feature_names=encoded_feature_names)


In [ ]:
# @@SUBSET@@ 3/3 — LORE sul campione a lesioni distinte
from isic24 import xai_driver as xd
import gc, keras, time, numpy as np

STRAT_LORE_ONLY_IMPL  = ["vanilla"] #None
STRAT_LORE_ONLY_SEEDS = [71]        # una run per famiglia; None = tutti i seed
STRAT_LORE_ONLY_TAG   = list(CENTRAL_SETTINGS.values())
STRAT_LORE_PER_FAMILY = 1
STRAT_LORE_DRY_RUN    = False
STRAT_LORE_N          = 48

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if STRAT_LORE_ONLY_IMPL:  _m = _m[_m.impl_name.isin(STRAT_LORE_ONLY_IMPL)]
if STRAT_LORE_ONLY_SEEDS: _m = _m[_m.seed.isin(STRAT_LORE_ONLY_SEEDS)]
if STRAT_LORE_ONLY_TAG:   _m = _m[_m.setting_tag.isin(STRAT_LORE_ONLY_TAG)]
if STRAT_LORE_PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(STRAT_LORE_PER_FAMILY))
_m = _m.reset_index(drop=True)
_RD = {r.run_id: os.path.join(ART_PATH, str(r.run_dir).split("artifacts/enhanced/")[-1])
       for _, r in _m.iterrows()}

# Il sottoinsieme da spiegare: meta' per classe, preso a passo costante DENTRO ogni classe
# cosi' non e' a sua volta un prefisso. `xai_subset_idx` e' ordinato e le labels stanno in
# validation_df alle stesse posizioni.
_y = validation_df.target.values[np.asarray(xai_subset_idx)]
_half = STRAT_LORE_N // 2
_pick = []
for _c in (0, 1):
    _pool = np.asarray(xai_subset_idx)[_y == _c]
    if len(_pool) < _half:
        raise ValueError(f"classe {_c}: {len(_pool)} disponibili, ne servono {_half}")
    _pick.append(_pool[np.linspace(0, len(_pool) - 1, _half).round().astype(int)])
_lore_idx = np.sort(np.concatenate(_pick))
_lore_y = validation_df.target.values[_lore_idx]
assert len(_lore_idx) == STRAT_LORE_N and _lore_y.sum() == _half, "il campione LORE non e' 50/50"
print(f"[subset-lore] campione: {len(_lore_idx)} lesioni, "
      f"{int(_lore_y.sum())} pos / {int((1-_lore_y).sum())} neg  (bilanciato)")

xd.LORE_SA_KWARGS["max_explain_samples"] = len(_lore_idx)

_todo, _done = xs.resume_report(_RD, "lore_sa", XAI_SUBSET_FP)
print(f"[subset-lore] {len(_m)} run | gia' fatte {len(_done)} | da fare {len(_todo)}")
MIN_PER_SAMPLE_LORE = 0.67       # misurato 30/07; vedi il commento in testa
print(f"[subset-lore] {len(_lore_idx)} campioni/run -> stima "
      f"{len(_todo)*len(_lore_idx)*MIN_PER_SAMPLE_LORE/60:.1f} h "
      f"({len(_lore_idx)*MIN_PER_SAMPLE_LORE:.0f} min/run)")
# Se una famiglia manca o e' rappresentata da un endpoint degenere, va visto ORA, non fra 3 ore.
print(f"[subset-lore] famiglie coperte: {_m.impl_name.nunique()}/6")
_deg = _m[_m.setting_tag.astype(str).str.contains("a0p000|lam0e\\+00|init_a0p100")]
if len(_deg):
    print(f"\n⚠ {len(_deg)} run selezionate sono setting DEGENERI (ramo tabellare azzerato o "
          f"lambda=0): controllo valido, NON rappresentanti di famiglia.")
    display(_deg[["impl_name", "setting_tag", "run_id"]])
display(_m[["impl_name", "setting_tag", "seed", "run_id"]])

if STRAT_LORE_DRY_RUN:
    print("\n*** DRY RUN — niente eseguito. STRAT_LORE_DRY_RUN=False per partire. ***")
else:
    _prev = xd.LORE_SA_KWARGS.get("subset_idx")
    xd.LORE_SA_KWARGS["subset_idx"] = _lore_idx
    _t0 = time.time()
    try:
        for _i, _r in _m.iterrows():
            _rd = _RD[_r.run_id]
            if xs.is_done(_rd, "lore_sa", XAI_SUBSET_FP):
                continue
            try:
                # overwrite=True disarma solo la scorciatoia di `xai_status.json`, che qui
                # mentirebbe: 29 run canoniche portano gia' `LoreSACallback: ok` dal giro del
                # 30/07, che scriveva in `xai/lore/epoch*/tabular_sa` — un'altra cartella.
                # Il write guard resta armato e rifiuta comunque di sovrascrivere un
                # `xai/lore_sa/` gia' pieno: "ricalcola" non vuol dire "puoi distruggere".
                _o = xd.run_posthoc_xai(_rd, engines=("lore",), best_epoch=int(_r.best_epoch),
                                        best_val_auc=float(_r.best_val_auc),
                                        overwrite=True, verbose=False)
                xs.mark_done(_rd, "lore_sa", XAI_SUBSET_FP,
                             {"run_id": _r.run_id, "impl_name": _r.impl_name,
                              "setting_tag": _r.setting_tag, "n": int(len(_lore_idx)),
                              "n_pos": int(_lore_y.sum()), "result": str(_o)})
                print(f"[{_i+1}/{len(_m)}] {_r.run_id}: {_o}  "
                      f"({(time.time()-_t0)/60:.0f} min totali)", flush=True)
            except Exception as _e:
                print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}", flush=True)
            finally:
                keras.backend.clear_session(); gc.collect()
    finally:
        xd.LORE_SA_KWARGS["subset_idx"] = _prev   # non lasciare il knob armato per le altre celle
    _todo, _done = xs.resume_report(_RD, "lore_sa", XAI_SUBSET_FP)
    print(f"\n[subset-lore] fatte {len(_done)}/{len(_m)} in {(time.time()-_t0)/60:.0f} min"
          + (f" — restano {len(_todo)}, rilancia per riprendere" if _todo else ""))
    if not _todo:
        xs.report(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                  ART_PATH, kinds=("lore_sa",), central=CENTRAL_SETTINGS,
                  encoded_feature_names=encoded_feature_names,
                  lore_idx={len(_lore_idx): _lore_idx})


## Appendice archiviata — verifiche Stage 0–3 (CONCLUSIONI RITIRATE)

Le verifiche Stage 0/1/2/2b/3 e i loop LORE post-hoc **non vivono più in questo notebook**:
erano 1599 righe (il 46% del file) di codice le cui conclusioni sono state ritirate.

Sono state spostate **verbatim** in **`scripts/archive_stage0_3.py`**, con l'indicazione
della cella di origine. Nulla è andato perso — il codice resta eseguibile (richiede la
sezione di setup di questo notebook già eseguita) e versionato.

**Due di quelle verifiche restano citate come fonti primarie nei report** e i loro
artefatti sono intatti:

| verifica | artefatto | citata in |
| --- | --- | --- |
| Stage 2 — attributore Permutation allineato | `runs/_robustness/stage2_aligned_attr/faithfulness_aligned_attr.csv` | REPORT_S §3.3, REPORT §7.5 |
| Stage 3 — robustezza immagine (subset + sweep σ) | `runs/_robustness/stage3_image/` | REPORT §7.4, §9 |

Perché sono state ritirate: `artifacts/enhanced/reports/FINDINGS_insertion_deletion_2026-07-21.md`.


In [ ]:
# @@SHAPE@@ — BACKFILL delle metriche di forma sugli artefatti vecchi (CPU, ~10 s)
# ================================================================================
# ⏳ CELLA TEMPORANEA — da cancellare quando ISIC24 e' chiuso. Serve SOLO perche' i 110 run
# di ISIC24 sono stati calcolati PRIMA che le metriche di forma entrassero nei motori.
#
# Da adesso in poi non serve piu' niente di tutto questo, e sui dataset nuovi non esistera':
#
#   `ShapCallback`     -> `shap_compactness`, `shap_k90_sparsity`, `shap_mass`,
#                          `frac_zero_shap` finiscono in `xai/shap/shap_summary.csv`
#                          mentre SHAP calcola.
#   `LoreSACallback`   -> `lore_sa_rule_compactness`, `lore_sa_frac_empty_rules` finiscono
#                          in `xai/lore_sa/lore_sa_per_epoch.csv` mentre LORE calcola.
#   `xs.report(...)`   -> stabilita' fra configurazioni e accordo SHAP<->LORE, in coda ai
#                          loop `2/3` e `3/3`. Sono cross-run: confrontano DUE run, quindi
#                          non possono uscire da un motore che ne vede una sola.
#
# Questa cella fa esattamente quel terzo passaggio, piu' la rilettura delle prime due dagli
# npz/CSV gia' scritti. Stessa funzione, stesso risultato.
from isic24 import xai_metrics as xm
import numpy as np, pandas as pd, os

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))

# Quali righe di validation_df ha spiegato LORE. Le run NUOVE lo scrivono da sole nel CSV
# (`val_index`); qui serve per gli artefatti vecchi, indicizzato sul numero di campioni —
# un array solo, applicato a run con campioni diversi, appaierebbe righe sbagliate.
LORE_IDX_BY_N = {
    48: np.asarray(xai_subset_idx)[:48],   # giro del 31/07: prefisso del campione a 144
    32: np.arange(32),                     # artefatti storici: prefisso grezzo
}

_out = xs.report(_m, ART_PATH, kinds=(), central=CENTRAL_SETTINGS,
                 encoded_feature_names=encoded_feature_names, lore_idx=LORE_IDX_BY_N)
_shape, _stab, _fam = _out["shape"], _out["stability"], _out["by_family"]
print(f"\n[forma] {len(_shape)} run | {len(_stab)} coppie | "
      f"{int(_shape.branch_off.sum())} run col ramo tabellare spento (metriche = NaN)")

# ---- T2: la stabilita' vale qualcosa solo CONTRO il rumore di seed --------------------
# "Stability@10 = 0.87 fra alpha=0.5 e alpha=0.625" da solo non e' interpretabile: puo'
# essere tutto rumore di inizializzazione. Il null e' la Jaccard fra due seed dello STESSO
# setting, sugli stessi campioni.
print("\n▶ T2 — Stability@k, e il suo null di seed")
display(_stab.pivot_table(index="impl_name", columns="pair_kind", values="stability_at_k",
                          aggfunc=["mean", "count"]).round(3))
print("  `adjacent_param` > `same_param_seed`  =>  spostare il peso della modalita' cambia")
print("  le top-k feature MENO di quanto le cambi il seed. Le famiglie senza asse di")
print("  parametro (vanilla, controlled_fusion, adaptive_gating: un solo setting) hanno")
print("  solo il null, ed e' corretto che sia cosi'.")

print("\n▶ Tutte le metriche affiancate — le `shape__` sono il complemento")
display(_fam.round(3))
print("""
COME SI LEGGE
  faith_*            fedelta' (primaria) — ordina le famiglie, e l'ordine e' la tesi
  lore_coverage      attendance (primaria) — quanto il modello guarda il ramo tabellare
  shape__*           leggibilita' (complemento) — NON si sommano alle precedenti
  shape__shap_mass   il controllo: sotto ~1.0 il ramo e' spento e la forma esce NaN
  shape__lore_n      se mostra piu' di un valore, quella famiglia ha run LORE su campioni
                     DIVERSI e le colonne LORE escono NaN invece di mediare cose diverse
  shape__stability_effect  = null_di_seed - stabilita' fra pesi vicini.
                     Negativo = il seed sposta le spiegazioni piu' del peso della modalita'.
""")
